# 13 · Classifieurs — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch13_classifieurs/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch13_classifieurs/03_notebook.ipynb`, créée par `python tools/start_chapter.py 13`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 13.Q1 à 13.Q12, 13.R1 à 13.R3, 13.1 à 13.7, 13.9 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★★ | |
| A | 13.12 | Prédire la frontière de cinq classifieurs avant de l'afficher | 🔮 | ★ | 15 |
|  | 13.13 | Choisir k : de la frontière en dents de scie à l'érosion | 🔬 | ★★ | 30 |
|  | 13.14 | Coder pairwise_distances et KNeighborsClassifier | 🔨 | ★★★ | 45 |
|  | 13.15 | kNN et malédiction de la dimension | 🔬 | ★★★ | 35 |
| B | 13.16 | SVM linéaire : vecteurs de support et paramètre C | 📦 | ★★ | 25 |
|  | 13.17 | Relever les cercles en 3D pour les séparer par un plan | 🎨 | ★★ | 30 |
|  | 13.18 | Noyau RBF : régler C et gamma par validation croisée | 📦 | ★★★ | 35 |
| C | 13.19 | Instabilité et profondeur d'un arbre | 🔬 | ★★ | 30 |
|  | 13.20 | Coder les impuretés et le gain d'un split | 🔨 | ★★ | 25 |
|  | 13.21 | L'arbre parfait qui triche : une colonne identifiant | 🐛 | ★★ | 20 |
|  | 13.22 | Coder la recherche du meilleur split | 🔨 | ★★★ | 45 |
|  | 13.23 | Arbres scikit-learn : plot_tree, export_text et élagage ccp_alpha | 📦 | ★★★ | 35 |
|  | 13.24 | Coder un arbre de décision récursif | 🔨 | ★★★★ | 150 |
|  | 13.25 | Préparer l'arbre pour les ensembles : poids, features tirées, importances | 🔨 | ★★★ | 50 |
|  | 13.26 | Arbre de régression : même squelette, autre critère | 🔨 | ★★ | 30 |
| D | 13.27 | Naïve Bayes sur des nuages gaussiens puis sur des lunes | 🔮 | ★ | 10 |
|  | 13.28 | Coder GaussianNB avec log-densités et logsumexp | 🔨 | ★★★ | 40 |
|  | 13.29 | Coder MultinomialNB avec lissage | 🔨 | ★★ | 30 |
|  | 13.30 | Dialogue ou narration ? Classer des phrases de Holmes : sac de mots, TF-IDF et Naïve Bayes | 📦 | ★★★ | 40 |
| E | 13.31 | Régression logistique scikit-learn : coefficients et probabilités | 📦 | ★★ | 25 |
|  | 13.32 | Coder la régression logistique par descente de gradient | 🔨 | ★★★ | 70 |
| F | 13.33 | Associer quatre frontières à leurs classifieurs | 📈 | ★ | 15 |
|  | 13.34 | Mesurer et vectoriser : accélérer son kNN | 🛠️ | ★★ | 25 |
|  | 13.35 | Comparatif mylearn contre scikit-learn : accuracy et temps | 🔬 | ★★★ | 45 |
|  | 13.36 | Défi : 96 % sur MNIST avec un classifieur classique | 🏆 | ★★★★ | 100 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="13")  # your own library

## Objectifs et rappel express

- Calculer à la main un vote de k plus proches voisins, une marge, une impureté, un gain de split et un posterior de Naïve Bayes.
- Écrire et vérifier contre scikit-learn les classifieurs de ta librairie : `KNeighborsClassifier`, les arbres de `mylearn.tree` (prêts pour les ensembles du ch. 14), `GaussianNB`, `MultinomialNB` et `LogisticRegression`.
- Régler $k$, C, $\gamma$ et la profondeur d'un arbre par validation croisée, et limiter l'overfitting d'un arbre.
- Comparer les classifieurs (accuracy, temps, mémoire, explicabilité) et choisir une baseline.

**Rappel express.** Distances : $\sqrt{\sum_j (x_j - z_j)^2}$ et $\sum_j \lvert x_j - z_j \rvert$ ; distance à un hyperplan : $\lvert \mathbf{w} \cdot \mathbf{x} + b \rvert / \lVert \mathbf{w} \rVert$, largeur de la bande $2/\lVert \mathbf{w} \rVert$ ; noyau RBF : $e^{-\gamma \lVert \mathbf{x} - \mathbf{z} \rVert^2}$ ; Gini : $1 - \sum_c p_c^2$ ; entropie : $-\sum_c p_c \log_2 p_c$ ; gain : $I(\text{parent}) - \frac{n_G}{n} I(G) - \frac{n_D}{n} I(D)$ ; Naïve Bayes : $\log P(c) + \sum_j \log P(x_j \mid c)$, normalisé par logsumexp ; lissage : $\frac{N_{cj} + \alpha}{N_c + \alpha d}$ ; régression logistique : $\hat p = \sigma(\mathbf{w} \cdot \mathbf{x} + b)$, gradient $\frac1n \mathbf{X}^\top (\hat{\mathbf{p}} - \mathbf{y})$. En Python : `sklearn.neighbors.KNeighborsClassifier`, `sklearn.svm.SVC`, `sklearn.tree` (`DecisionTreeClassifier`, `plot_tree`, `export_text`), `sklearn.naive_bayes` (`GaussianNB`, `MultinomialNB`), `sklearn.linear_model.LogisticRegression`, `sklearn.model_selection.GridSearchCV`.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 13.1 à 13.7 et 🧮 13.9)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; une suite de codes : les lettres dans l'ordre (`"XYZ"`) ; plusieurs nombres : une liste (`[4, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la preuve ∂ 13.8, la réflexion (🗣️ 13.10, ⚖️ 13.11) et l'entretien se corrigent avec `05_solutions.md`.

**13.Q1 — Pourquoi essayer des classifieurs classiques d'abord ?**

In [ ]:
# 13.Q1: the values you found on paper
answer_13_Q1a = ...  # a) True or False
answer_13_Q1b = ...  # b) the letter of your choice, e.g. "E"
answer_13_Q1c = ...  # c) True or False
answer_13_Q1d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_13_Q1a, answer_13_Q1b, answer_13_Q1c, answer_13_Q1d]):
    wb.check(f"13.Q1{letter}", answer)

**13.Q2 — Paramétrique ou non paramétrique ?**

In [ ]:
# 13.Q2: the values you found on paper
answer_13_Q2a = ...  # a) six letters P or N, in the order of the items, e.g. "XYXYXY"
answer_13_Q2b = ...  # b) the letter of your choice
answer_13_Q2c = ...  # c) True or False
answer_13_Q2d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_13_Q2a, answer_13_Q2b, answer_13_Q2c, answer_13_Q2d]):
    wb.check(f"13.Q2{letter}", answer)

**13.Q3 — kNN n'est pas k-means**

In [ ]:
# 13.Q3: the values you found on paper
answer_13_Q3a = ...  # a) the letter of your choice
answer_13_Q3b = ...  # b) the letter of your choice
answer_13_Q3c = ...  # c) True or False
answer_13_Q3d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_13_Q3a, answer_13_Q3b, answer_13_Q3c, answer_13_Q3d]):
    wb.check(f"13.Q3{letter}", answer)

**13.Q4 — Petit k, grand k**

In [ ]:
# 13.Q4: the values you found on paper
answer_13_Q4a = ...  # a) a number
answer_13_Q4b = ...  # b) the letter of your choice
answer_13_Q4c = ...  # c) the letter of your choice
answer_13_Q4d = ...  # d) True or False
answer_13_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_13_Q4a, answer_13_Q4b, answer_13_Q4c, answer_13_Q4d, answer_13_Q4e]):
    wb.check(f"13.Q4{letter}", answer)

**13.Q5 — Marge, vecteurs de support et C**

In [ ]:
# 13.Q5: the values you found on paper
answer_13_Q5a = ...  # a) True or False
answer_13_Q5b = ...  # b) the letter of your choice
answer_13_Q5c = ...  # c) the letter of your choice
answer_13_Q5d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_13_Q5a, answer_13_Q5b, answer_13_Q5c, answer_13_Q5d]):
    wb.check(f"13.Q5{letter}", answer)

**13.Q6 — Ce que l'astuce du noyau évite de calculer**

In [ ]:
# 13.Q6: the values you found on paper
answer_13_Q6a = ...  # a) the letter of your choice
answer_13_Q6b = ...  # b) a whole number
answer_13_Q6c = ...  # c) a number
answer_13_Q6d = ...  # d) True or False
answer_13_Q6e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_13_Q6a, answer_13_Q6b, answer_13_Q6c, answer_13_Q6d, answer_13_Q6e]):
    wb.check(f"13.Q6{letter}", answer)

**13.Q7 — Racine, feuilles, profondeur : le vocabulaire des arbres**

In [ ]:
# 13.Q7: the values you found on paper
answer_13_Q7a = ...  # a) a whole number
answer_13_Q7b = ...  # b) a whole number
answer_13_Q7c = ...  # c) a whole number
answer_13_Q7d = ...  # d) a whole number
answer_13_Q7e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_13_Q7a, answer_13_Q7b, answer_13_Q7c, answer_13_Q7d, answer_13_Q7e]):
    wb.check(f"13.Q7{letter}", answer)

**13.Q8 — Pourquoi un arbre surapprend-il ?**

In [ ]:
# 13.Q8: the values you found on paper
answer_13_Q8a = ...  # a) a number
answer_13_Q8b = ...  # b) True or False
answer_13_Q8c = ...  # c) the letter of your choice
answer_13_Q8d = ...  # d) the letter of your choice

for letter, answer in zip("abcd", [answer_13_Q8a, answer_13_Q8b, answer_13_Q8c, answer_13_Q8d]):
    wb.check(f"13.Q8{letter}", answer)

**13.Q9 — Gini ou gain d'information ?**

In [ ]:
# 13.Q9: the values you found on paper
answer_13_Q9a = ...  # a) a number
answer_13_Q9b = ...  # b) a number
answer_13_Q9c = ...  # c) a number
answer_13_Q9d = ...  # d) the letter of your choice
answer_13_Q9e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_13_Q9a, answer_13_Q9b, answer_13_Q9c, answer_13_Q9d, answer_13_Q9e]):
    wb.check(f"13.Q9{letter}", answer)

**13.Q10 — Limiter la profondeur ou élaguer ?**

In [ ]:
# 13.Q10: the values you found on paper
answer_13_Q10a = ...  # a) five letters P or E, in the order of the items, e.g. "XYXYX"
answer_13_Q10b = ...  # b) True or False
answer_13_Q10c = ...  # c) the letter of your choice

for letter, answer in zip("abc", [answer_13_Q10a, answer_13_Q10b, answer_13_Q10c]):
    wb.check(f"13.Q10{letter}", answer)

**13.Q11 — Ce qui est vraiment « naïf » dans Naïve Bayes**

In [ ]:
# 13.Q11: the values you found on paper
answer_13_Q11a = ...  # a) the letter of your choice
answer_13_Q11b = ...  # b) True or False
answer_13_Q11c = ...  # c) a whole number
answer_13_Q11d = ...  # d) True or False
answer_13_Q11e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_13_Q11a, answer_13_Q11b, answer_13_Q11c, answer_13_Q11d, answer_13_Q11e]):
    wb.check(f"13.Q11{letter}", answer)

**13.Q12 — Quel classifieur pour quelle situation ?**

In [ ]:
# 13.Q12: the values you found on paper
answer_13_Q12a = ...  # a) five letters A, N, K, L, S, in the order of the situations, e.g. "AAAAA"
answer_13_Q12b = ...  # b) True or False

for letter, answer in zip("ab", [answer_13_Q12a, answer_13_Q12b]):
    wb.check(f"13.Q12{letter}", answer)

**13.R1 — Ch. 12 : pourquoi standardiser avant de mesurer des distances**

In [ ]:
# 13.R1: the values you found on paper
answer_13_R1a = ...  # a) a number with 2 decimals
answer_13_R1b = ...  # b) a number with 2 decimals
answer_13_R1c = ...  # c) the letter "A" or "B"
answer_13_R1d = ...  # d) a number with 3 decimals
answer_13_R1e = ...  # e) the letter "A" or "B"
answer_13_R1f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_13_R1a, answer_13_R1b, answer_13_R1c, answer_13_R1d, answer_13_R1e, answer_13_R1f]):
    wb.check(f"13.R1{letter}", answer)

**13.R2 — Ch. 10 : l'équation de la frontière d'un perceptron**

In [ ]:
# 13.R2: the values you found on paper
answer_13_R2a = ...  # a) 0 or 1
answer_13_R2b = ...  # b) 0 or 1
answer_13_R2c = ...  # c) the letter of your choice
answer_13_R2d = ...  # d) a number
answer_13_R2e = ...  # e) a number
answer_13_R2f = ...  # f) a number with 3 decimals
answer_13_R2g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_13_R2a, answer_13_R2b, answer_13_R2c, answer_13_R2d, answer_13_R2e, answer_13_R2f, answer_13_R2g]):
    wb.check(f"13.R2{letter}", answer)

**13.R3 — Ch. 6 : entropie de deux distributions**

In [ ]:
# 13.R3: the values you found on paper
answer_13_R3a = ...  # a) a number
answer_13_R3b = ...  # b) a number with 3 decimals
answer_13_R3c = ...  # c) a number
answer_13_R3d = ...  # d) a number with 3 decimals
answer_13_R3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_13_R3a, answer_13_R3b, answer_13_R3c, answer_13_R3d, answer_13_R3e]):
    wb.check(f"13.R3{letter}", answer)

**Ex 13.1 — Vote des k plus proches voisins à la main**

In [ ]:
# Ex 13.1: the values you found on paper
answer_13_1a = ...  # a) an index from 0 to 6
answer_13_1b = ...  # b) a class: 0, 1 or 2
answer_13_1c = ...  # c) a class: 0, 1 or 2
answer_13_1d = ...  # d) a number with 3 decimals
answer_13_1e = ...  # e) a class: 0, 1 or 2
answer_13_1f = ...  # f) a class: 0, 1 or 2
answer_13_1g = ...  # g) a number with 3 decimals
answer_13_1h = ...  # h) a class: 0, 1 or 2
answer_13_1i = ...  # i) a list [index, class]

for letter, answer in zip("abcdefghi", [answer_13_1a, answer_13_1b, answer_13_1c, answer_13_1d, answer_13_1e, answer_13_1f, answer_13_1g, answer_13_1h, answer_13_1i]):
    wb.check(f"13.1{letter}", answer)

**Ex 13.2 — Gini et entropie d'une feuille**

In [ ]:
# Ex 13.2: the values you found on paper
answer_13_2a = ...  # a) a number
answer_13_2b = ...  # b) a number with 3 decimals
answer_13_2c = ...  # c) a number
answer_13_2d = ...  # d) a number
answer_13_2e = ...  # e) a number with 3 decimals
answer_13_2f = ...  # f) a number
answer_13_2g = ...  # g) a number with 3 decimals
answer_13_2h = ...  # h) a number with 3 decimals
answer_13_2i = ...  # i) a number with 3 decimals

for letter, answer in zip("abcdefghi", [answer_13_2a, answer_13_2b, answer_13_2c, answer_13_2d, answer_13_2e, answer_13_2f, answer_13_2g, answer_13_2h, answer_13_2i]):
    wb.check(f"13.2{letter}", answer)

**Ex 13.3 — Choisir le meilleur seuil sur la longueur des nageoires**

In [ ]:
# Ex 13.3: the values you found on paper
answer_13_3a = ...  # a) a whole number
answer_13_3b = ...  # b) a number
answer_13_3c = ...  # c) a number
answer_13_3d = ...  # d) a number with 3 decimals
answer_13_3e = ...  # e) a number with 3 decimals
answer_13_3f = ...  # f) a threshold in mm
answer_13_3g = ...  # g) a number with 3 decimals
answer_13_3h = ...  # h) a whole number
answer_13_3i = ...  # i) 0 (Adélie) or 1 (Papou)
answer_13_3j = ...  # j) a whole number

for letter, answer in zip("abcdefghij", [answer_13_3a, answer_13_3b, answer_13_3c, answer_13_3d, answer_13_3e, answer_13_3f, answer_13_3g, answer_13_3h, answer_13_3i, answer_13_3j]):
    wb.check(f"13.3{letter}", answer)

**Ex 13.4 — Distance à une droite et marge d'un SVM**

In [ ]:
# Ex 13.4: the values you found on paper
answer_13_4a = ...  # a) a number with 4 decimals
answer_13_4b = ...  # b) a number
answer_13_4c = ...  # c) a number with 3 decimals
answer_13_4d = ...  # d) a number
answer_13_4e = ...  # e) a list of indices, in increasing order
answer_13_4f = ...  # f) a number with 3 decimals
answer_13_4g = ...  # g) +1 or -1
answer_13_4h = ...  # h) True or False
answer_13_4i = ...  # i) True or False
answer_13_4j = ...  # j) True or False

for letter, answer in zip("abcdefghij", [answer_13_4a, answer_13_4b, answer_13_4c, answer_13_4d, answer_13_4e, answer_13_4f, answer_13_4g, answer_13_4h, answer_13_4i, answer_13_4j]):
    wb.check(f"13.4{letter}", answer)

**Ex 13.5 — L'astuce du noyau vérifiée à la main**

In [ ]:
# Ex 13.5: the values you found on paper
answer_13_5a = ...  # a) a number
answer_13_5b = ...  # b) a number
answer_13_5c = ...  # c) a number with 3 decimals
answer_13_5d = ...  # d) a number
answer_13_5e = ...  # e) a number
answer_13_5f = ...  # f) a number with 4 decimals
answer_13_5g = ...  # g) a number with 2 decimals
answer_13_5h = ...  # h) a whole number

for letter, answer in zip("abcdefgh", [answer_13_5a, answer_13_5b, answer_13_5c, answer_13_5d, answer_13_5e, answer_13_5f, answer_13_5g, answer_13_5h]):
    wb.check(f"13.5{letter}", answer)

**Ex 13.6 — Anglais ou français ? Naïve Bayes multinomial sur des lettres**

In [ ]:
# Ex 13.6: the values you found on paper
answer_13_6a = ...  # a) a number
answer_13_6b = ...  # b) a number with 3 decimals
answer_13_6c = ...  # c) a number with 4 decimals
answer_13_6d = ...  # d) a number with 4 decimals
answer_13_6e = ...  # e) a number with 3 decimals
answer_13_6f = ...  # f) a number with 3 decimals
answer_13_6g = ...  # g) a number with 3 decimals
answer_13_6h = ...  # h) a number
answer_13_6i = ...  # i) a number with 3 decimals

for letter, answer in zip("abcdefghi", [answer_13_6a, answer_13_6b, answer_13_6c, answer_13_6d, answer_13_6e, answer_13_6f, answer_13_6g, answer_13_6h, answer_13_6i]):
    wb.check(f"13.6{letter}", answer)

**Ex 13.7 — Posterior d'un Naïve Bayes gaussien**

In [ ]:
# Ex 13.7: the values you found on paper
answer_13_7a = ...  # a) a number
answer_13_7b = ...  # b) a number
answer_13_7c = ...  # c) a number with 3 decimals
answer_13_7d = ...  # d) a number with 3 decimals
answer_13_7e = ...  # e) a number with 3 decimals
answer_13_7f = ...  # f) a number with 3 decimals
answer_13_7g = ...  # g) a number with 3 decimals
answer_13_7h = ...  # h) a number with 3 decimals
answer_13_7i = ...  # i) a number with 3 decimals
answer_13_7j = ...  # j) a number with 3 decimals
answer_13_7k = ...  # k) True or False

for letter, answer in zip("abcdefghijk", [answer_13_7a, answer_13_7b, answer_13_7c, answer_13_7d, answer_13_7e, answer_13_7f, answer_13_7g, answer_13_7h, answer_13_7i, answer_13_7j, answer_13_7k]):
    wb.check(f"13.7{letter}", answer)

**Ex 13.9 — Mémoire et temps d'un kNN sur MNIST complet**

In [ ]:
# Ex 13.9: the values you found on paper
answer_13_9a = ...  # a) a whole number
answer_13_9b = ...  # b) a number with 2 decimals (MB)
answer_13_9c = ...  # c) a number with 2 decimals (MB)
answer_13_9d = ...  # d) a whole number
answer_13_9e = ...  # e) a number with 2 decimals (seconds)
answer_13_9f = ...  # f) a number (GB)
answer_13_9g = ...  # g) a number (MB)
answer_13_9h = ...  # h) a whole number
answer_13_9i = ...  # i) a whole number (rounded)
answer_13_9j = ...  # j) a number (MB)

for letter, answer in zip("abcdefghij", [answer_13_9a, answer_13_9b, answer_13_9c, answer_13_9d, answer_13_9e, answer_13_9f, answer_13_9g, answer_13_9h, answer_13_9i, answer_13_9j]):
    wb.check(f"13.9{letter}", answer)

## Partie A · Les k plus proches voisins

Cinq classifieurs sur un « ou exclusif » bruité, pour fixer les idées (13.12) ; puis le kNN : le choix de $k$ (13.13), ta classe `KNeighborsClassifier` (13.14) et ce que deviennent les distances quand la dimension augmente (13.15). La cellule suivante importe les classifieurs de scikit-learn, définit les outils des parties A à F et découpe les 333 manchots complets (233 pour l'entraînement, 100 pour le test), dont la partie 0 n'a utilisé que quelques valeurs. Ne recopie jamais un nombre à la main : chaque réponse est calculée par ton code, et pas arrondie.

In [ ]:
# Tools for the notebook exercises (parts A to F), and the penguins
import contextlib
import io
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from wb.errors import MylearnMissingError


def verdict(ex_id, ok, success, failure):
    """Print ✅ with the success message, or ❌ with the failure message."""
    print(f"✅ Ex {ex_id} : {success}" if ok else f"❌ Ex {ex_id} : {failure}")


def filled(*values):
    """True when none of the values is still `...` (or None): the answer has been written."""
    return all(value is not ... and value is not None for value in values)


def fr(x, digits=3):
    """A number written the French way, with a decimal comma (for the messages in French)."""
    return f"{x:.{digits}f}".replace(".", ",")


def returned(ex_id, name, value):
    """False, with a ❌ message, when the function `name` returned None (a forgotten return); True otherwise."""
    if value is None:
        print(f"❌ Ex {ex_id} : {name} renvoie None : as-tu oublié le return ?")
        return False
    return True


def print_answer(ex_id, value, **_):
    """Solutions notebook: show the value that the exercise notebook checks with wb.check."""
    try:
        shown = np.round(np.asarray(value, dtype=float), 4).tolist() if isinstance(value, (np.ndarray, list, tuple)) else value
    except (TypeError, ValueError):     # a list of names
        shown = list(value)
    print(f"{ex_id}:", shown)


def fitted(model, X, y=None, **fit_params):
    """The model after model.fit(X, y): no chained call, so that a fit that forgets `return self` (the tests report
    it) does not stop the check cells."""
    if y is None:
        model.fit(X, **fit_params)
    else:
        model.fit(X, y, **fit_params)
    return model


@contextlib.contextmanager
def checked(ex_id):
    """Like wb.attempt (⏳ while the code is not written yet), but any other error raised by YOUR code becomes one ❌
    line instead of stopping the cell, so that the tests run after it can say more."""
    with wb.attempt(ex_id):
        try:
            yield
        except (NotImplementedError, ImportError, MylearnMissingError):
            raise
        except Exception as exc:                              # a bug in the learner's code: report it, go on
            if isinstance(exc, TypeError) and "NoneType" in str(exc):
                raise                                         # wb.attempt explains the forgotten return
            hint = " ; une fonction ou une méthode oublie-t-elle son return ?" if "NoneType" in str(exc) else ""
            print(f"❌ Ex {ex_id} : ton code lève {type(exc).__name__} : {str(exc)[:300]}{hint}")


def written(probe):
    """False while probe() stops on code not written yet (NotImplementedError, or no mylearn at all); True otherwise,
    even when it raises another error (the tests then say which)."""
    try:
        probe()
    except (NotImplementedError, ImportError, MylearnMissingError):
        return False
    except Exception:
        return True
    return True


TEST_FILES = {module: f"tests/test_ch13_{module}.py" for module in ["neighbors", "tree", "naive_bayes", "logistic"]}


def run_mylearn_tests(module, keyword, impl="learner"):
    """Run the tests of mylearn.<module> selected by `keyword` (on YOUR code by default)."""
    test_file = TEST_FILES[module]
    command = [sys.executable, "-m", "pytest", test_file, "-k", keyword, "-q", "-p", "no:cacheprovider",
               "--color=no", "-rf", "--tb=no"]
    if impl != "learner":
        command.append(f"--impl={impl}")
    result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True,
                            env={**os.environ, "COLUMNS": "1000"})   # long lines: the reason of each failure
    lines = result.stdout.strip().splitlines()
    failed = [line for line in lines if line.startswith("FAILED ")]
    if any("'NoneType' object has no attribute" in line for line in failed):
        print("💡 un attribut est lu sur None : une méthode oublie-t-elle son return ?")
    for line in failed[:8]:                                  # the test, then the reason of its failure
        name, _, reason = line.removeprefix(f"FAILED {test_file}::").partition(" - ")
        print(f"❌ {name}\n   {reason[:800]}")
    if len(failed) > 8:
        print(f"   ... and {len(failed) - 8} other failed test(s)")
    print("pytest:", lines[-1] if lines else result.stderr.strip()[-300:])


PENGUINS = wb.datasets.load_penguins(dropna=True).reset_index(drop=True)   # the 333 complete penguins
MEASURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
P_TRAIN, P_TEST = train_test_split(PENGUINS, test_size=100, random_state=1314, stratify=PENGUINS["species"])
print(len(P_TRAIN), "training penguins,", len(P_TEST), "test penguins")

### Ex 13.12 — Prédire la frontière de cinq classifieurs avant de l'afficher 🔮 ★ ⏱️ 15 min
**Objectif :** prévoir, avant de les voir, les scores de cinq classifieurs sur un problème qu'une droite ne sépare pas.  
**Prérequis :** ch. 7 (classification) · fiche §13.1, §13.2 et §13.7 (tableau « frontière ») · **Fil rouge :** synthétique · **Parcours :** R, C

`X_12` : 600 points bruités autour des quatre coins du carré unité ; la classe vaut 1 quand une seule des deux coordonnées est proche de 1 (un « ou exclusif »), 0 sinon. On entraîne sur 70 % des points, avec leurs réglages par défaut, cinq classifieurs : A `KNeighborsClassifier(n_neighbors=15)` ; B `SVC()` (noyau RBF) ; C `DecisionTreeClassifier()`, sans limite ; D `GaussianNB()` ; E `LogisticRegression()`. La fiche (tableau du §13.7) te dit déjà la **forme** de leurs frontières ; prédis plutôt des nombres.

Prédis, **avant** d'exécuter quoi que ce soit :
a) `chance_12` : les lettres des classifieurs dont l'accuracy de test reste **sous 65 %**, dans n'importe quel ordre (`""` si aucun) ;
b) `leaves_12` : le nombre de feuilles de l'arbre C : `"A"` moins de 10 ; `"B"` de 10 à 50 ; `"C"` de 50 à 150 ; `"D"` plus de 150 (une feuille par point d'entraînement en ferait 420) ;
c) `tree_beats_knn_12` : l'arbre C a-t-il une meilleure accuracy de test que le kNN A ? (`True` ou `False`)

Puis exécute l'expérience : elle affiche les accuracies d'entraînement et de test, et les cinq frontières sur les points de test.

In [ ]:
X_12, Y_12 = wb.synth.make_xor(n=600, noise=0.25, seed=1312)   # 4 noisy corners, class = x1 XOR x2
X_TRAIN_12, X_TEST_12, Y_TRAIN_12, Y_TEST_12 = train_test_split(X_12, Y_12, test_size=0.3, random_state=1312,
                                                                stratify=Y_12)
print(len(X_TRAIN_12), "training points,", len(X_TEST_12), "test points")

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
chance_12 = ...          # a) the letters, e.g. "AB", or "" if none
leaves_12 = ...          # b) "A", "B", "C" or "D"
tree_beats_knn_12 = ...  # c) True or False

In [ ]:
if any(answer is ... or answer is None for answer in [chance_12, leaves_12, tree_beats_knn_12]):
    print("⏳ Ex 13.12 : écris d'abord tes trois prédictions, puis relance cette cellule.")
else:
    models_12 = {"A": KNeighborsClassifier(n_neighbors=15), "B": SVC(), "C": DecisionTreeClassifier(random_state=0),
                 "D": GaussianNB(), "E": LogisticRegression()}
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    for ax, (letter, model) in zip(axes, models_12.items()):
        model.fit(X_TRAIN_12, Y_TRAIN_12)
        leaves = f" · {model.get_n_leaves()} leaves" if letter == "C" else ""
        print(f"{letter} {type(model).__name__:24s} training {model.score(X_TRAIN_12, Y_TRAIN_12):.3f} · "
              f"test {model.score(X_TEST_12, Y_TEST_12):.3f}{leaves}")
        wb.plot.plot_decision_boundary(model, X_TEST_12, Y_TEST_12, ax=ax, title=f"{letter} · {type(model).__name__}")
    plt.show()
    wb.check("13.12a", chance_12)
    wb.check("13.12b", leaves_12)
    wb.check("13.12c", tree_beats_knn_12)

Dans tes notes : compare avec tes prédictions. Pour chaque classifieur dont l'accuracy t'a surpris, cherche ce qu'il sait représenter et ce qu'il ne sait pas. Où l'arbre place-t-il ses premières questions (`export_text`) ? Explique son nombre de feuilles.

### Ex 13.13 — Choisir k : de la frontière en dents de scie à l'érosion 🔬 ★★ ⏱️ 30 min
**Objectif :** choisir $k$ par validation croisée, et voir ce qu'un grand $k$ efface.  
**Prérequis :** Ex 13.12 · ch. 8 (validation croisée) · fiche §13.3 · **Fil rouge :** synthétique · **Parcours :** R, C

`X_13` : 600 points de deux demi-lunes bruitées, plus une petite **île** de 15 points de la classe 1 autour de `ISLAND_13` = (−0,2 ; 1,0), au milieu de la lune du haut (classe 0). Avec le `KNeighborsClassifier` de scikit-learn (ta classe arrive en 13.14), calcule :
a) `train_k1_13` : l'accuracy d'un kNN à $k = 1$ entraîné sur `X_13` et mesurée sur ces **mêmes** points ;
b) `cv_13` : la liste des accuracies moyennes en validation croisée à 5 folds (`CV_13`), une par $k$ de `KS_13` ;
c) `best_k_13` : le $k$ de `KS_13` qui a la meilleure accuracy en b) ;
d) `erased_13` : le plus petit $k$ de `KS_13` pour lequel un kNN entraîné sur tous les points prédit la classe 0 en `ISLAND_13` (l'île a disparu).

La vérification contrôle a) à d), puis trace les frontières pour $k$ = 1, 15 et 101, l'île entourée.

Dans tes notes : pourquoi a) ne dit-il rien de la qualité du modèle ? L'île est-elle un vrai phénomène ou du bruit, et la validation croisée peut-elle le dire ? Que gagne-t-on, et que perd-on, en passant de $k = 1$ à $k = 101$ ?

In [ ]:
X_MOONS_13, Y_MOONS_13 = wb.synth.make_moons(n=600, noise=0.25, seed=1313)
ISLAND_13 = np.array([-0.2, 1.0])                  # the centre of a small region of class 1 in the upper moon (class 0)
island_points_13 = np.random.default_rng(1313).normal(ISLAND_13, 0.08, (15, 2))
X_13 = np.vstack([X_MOONS_13, island_points_13])   # 615 points
Y_13 = np.concatenate([Y_MOONS_13, np.ones(15, dtype=int)])
KS_13 = [1, 3, 5, 9, 15, 25, 41, 61, 101, 151]
CV_13 = StratifiedKFold(n_splits=5, shuffle=True, random_state=1313)

In [ ]:
train_k1_13 = ...   # a) accuracy of KNeighborsClassifier(n_neighbors=1) fitted on X_13, Y_13, measured on X_13, Y_13
cv_13 = ...         # b) list: for each k of KS_13, the mean accuracy of a kNN in 5-fold cross-validation (CV_13)
best_k_13 = ...     # c) the k of KS_13 with the best mean accuracy in b)
erased_13 = ...     # d) the smallest k of KS_13 whose kNN fitted on X_13, Y_13 predicts class 0 at ISLAND_13

In [ ]:
wb.check("13.13a", train_k1_13)
wb.check("13.13b", cv_13)
wb.check("13.13c", best_k_13)
wb.check("13.13d", erased_13)
with wb.attempt("13.13"):
    if not filled(cv_13, erased_13):
        raise NotImplementedError("cv_13, erased_13")
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, k in zip(axes, [1, 15, 101]):
        model = KNeighborsClassifier(n_neighbors=k).fit(X_13, Y_13)
        wb.plot.plot_decision_boundary(model, X_13, Y_13, ax=ax, title=f"k = {k}")
        ax.add_patch(plt.Circle(ISLAND_13, 0.25, fill=False, color="black", ls="--"))
    plt.show()

### Ex 13.14 — Coder pairwise_distances et KNeighborsClassifier 🔨 ★★★ ⏱️ 45 min
**Objectif :** écrire un kNN en force brute, vectorisé, qui donne exactement les votes de scikit-learn.  
**Prérequis :** ✏️ 13.1 · ch. 12 (standardisation) · 0A (classes Python) · fiche §13.3 · **Fil rouge :** Penguins · **mylearn :** `neighbors.py` · **Parcours :** R, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/neighbors.py` (créé par `python tools/start_chapter.py 13`), lis la docstring de chaque fonction et de chaque méthode, remplace les `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy seulement : scikit-learn et SciPy sont les **oracles** des tests. La cellule de vérification recharge ta librairie, compare quelques résultats à scikit-learn, puis lance les tests de l'exercice ; `python -m pytest tests/test_ch13_neighbors.py -q` les lance tous dans un terminal.

Écris `pairwise_distances` et la classe `KNeighborsClassifier` de `neighbors.py`. **Contrat** (les docstrings, plus ces précisions que vérifient les tests) :
- `pairwise_distances(A, B, metric="euclidean")` reçoit deux tableaux 2-D (ou des listes) de mêmes nombres de colonnes et rend la matrice `float64` de forme `(len(A), len(B))` des distances `"euclidean"` ou `"manhattan"`, **sans boucle Python sur les exemples** ; la distance d'un point à lui-même vaut **exactement** 0 (les tests le vérifient) ; elle ne modifie pas ses arguments ; `ValueError` si `A` ou `B` n'est pas 2-D, si leurs nombres de colonnes diffèrent, si la métrique est inconnue ou si une valeur est `NaN` (comme scikit-learn) ;
- `__init__(n_neighbors=5, weights="uniform", metric="euclidean")` range ses trois arguments tels quels, sans rien vérifier ni calculer ;
- `fit(X, y)` vérifie, puis **range des copies** de `X` et `y` dans des attributs dont le nom finit par `_` (ou commence par `_`), avec `classes_` (les labels distincts triés), `n_features_in_` et `n_samples_fit_`, et renvoie `self` ; un second `fit` oublie le premier ; `ValueError` si `n_neighbors < 1` ou `n_neighbors > len(X)` (dès `fit`), si `weights` ou `metric` est inconnu, si `X` n'est pas 2-D ou contient `NaN`, ou si `len(X) != len(y)` ;
- `kneighbors(X, n_neighbors=None, return_distance=True)` rend `(dist, ind)` (ou `ind` seul), de forme `(len(X), k)`, du plus proche au plus lointain ; à distances égales, le plus petit indice d'entraînement d'abord (un tri **stable**) ; `RuntimeError` avant `fit` ; `ValueError` pour une requête qui n'est pas 2-D (une seule requête s'écrit `X[None, :]`), un mauvais nombre de colonnes, une valeur `NaN`, ou un $k$ hors de 1 à `n_samples_fit_` ;
- `predict_proba(X)` rend la part (pondérée) des votes de chaque classe, colonne $j$ pour `classes_[j]` ; avec `weights="distance"`, un voisin à distance $d$ pèse $1/d$, et des voisins à distance 0 se partagent tout le poids ; `predict(X)` rend la classe de plus grande part (à égalité, la première de `classes_`) ; `score(X, y)` rend l'accuracy, un `float` Python.

La vérification essaie l'exemple de la docstring, compare tes probabilités et tes prédictions à celles de scikit-learn sur les manchots standardisés (deux réglages), puis lance les tests.

Dans tes notes : que coûte un appel à `predict` sur 100 manchots, en nombre de distances ? Pourquoi un tri stable, et pourquoi conseille-t-on souvent un $k$ impair à deux classes ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="13")   # reload your saved file
with checked("13.14"):
    D_doc_14 = mylearn.neighbors.pairwise_distances(np.array([[0.0, 0.0], [1.0, 1.0]]), np.array([[3.0, 4.0]]))
    if returned("13.14", "pairwise_distances", D_doc_14):
        print("docstring example:", np.round(np.asarray(D_doc_14, dtype=float), 4).ravel().tolist(), "(expected [5.0, 3.6056])")
with checked("13.14"):
    scaler_14 = StandardScaler().fit(P_TRAIN[MEASURES])
    Z_train_14, Z_test_14 = scaler_14.transform(P_TRAIN[MEASURES]), scaler_14.transform(P_TEST[MEASURES])
    y_train_14, y_test_14 = P_TRAIN["species"].to_numpy(), P_TEST["species"].to_numpy()
    for k, weights, metric in [(5, "uniform", "euclidean"), (6, "distance", "manhattan")]:
        setting_14 = f"k = {k}, {weights}, {metric}"
        mine_14 = fitted(mylearn.neighbors.KNeighborsClassifier(n_neighbors=k, weights=weights, metric=metric),
                         Z_train_14, y_train_14)
        oracle_14 = KNeighborsClassifier(n_neighbors=k, weights=weights, metric=metric, algorithm="brute")
        oracle_14.fit(Z_train_14, y_train_14)
        proba_14 = mine_14.predict_proba(Z_test_14)
        if not returned("13.14", "predict_proba", proba_14):
            continue
        predicted_14 = mine_14.predict(Z_test_14)
        if not returned("13.14", "predict", predicted_14):
            continue
        proba_14, predicted_14 = np.asarray(proba_14), np.asarray(predicted_14)
        if proba_14.shape != (len(Z_test_14), 3):
            print(f"❌ Ex 13.14 : {setting_14} : predict_proba devrait rendre un tableau (100, 3), une colonne par "
                  f"espèce de classes_, pas un tableau de forme {proba_14.shape}.")
        elif not np.allclose(proba_14, oracle_14.predict_proba(Z_test_14)):
            print(f"❌ Ex 13.14 : {setting_14} : tes probabilités diffèrent de celles de scikit-learn ; les tests "
                  "ci-dessous disent où.")
        elif not np.array_equal(predicted_14, oracle_14.predict(Z_test_14)):
            print(f"❌ Ex 13.14 : {setting_14} : tes probabilités sont justes, mais predict rend autre chose que les "
                  f"labels de scikit-learn (par exemple {predicted_14[:3].tolist()} au lieu de "
                  f"{oracle_14.predict(Z_test_14)[:3].tolist()}).")
        else:
            print(f"✅ Ex 13.14 : {setting_14} : mêmes probabilités et mêmes prédictions que scikit-learn sur les 100 "
                  f"manchots de test (accuracy {fr(oracle_14.score(Z_test_14, y_test_14))}).")
if written(lambda: mylearn.neighbors.pairwise_distances([[0.0]], [[1.0]])):
    run_mylearn_tests("neighbors", keyword="test_pairwise_distances_ or test_kneighbors_classifier_")
else:
    print("⏳ Ex 13.14 : les tests attendent ton pairwise_distances.")

### Ex 13.15 — kNN et malédiction de la dimension 🔬 ★★★ ⏱️ 35 min
**Objectif :** mesurer comment des features inutiles noient les distances d'un kNN, et pourquoi.  
**Prérequis :** Ex 13.14 · ch. 7 (malédiction de la dimension) · ch. 12 (standardisation) · fiche §13.3 · **Fil rouge :** Penguins · **Parcours :** C

Deux expériences avec **ta** librairie (`mylearn.neighbors` ; si 13.14 n'est pas fait, la référence prend le relais). Écris :
- `noisy_accuracy_15(n_noise, seed)` : l'accuracy de test de ton `KNeighborsClassifier(n_neighbors=5)` entraîné sur les 4 mesures standardisées des manchots (`Z_TRAIN_15`) **plus** `n_noise` colonnes de bruit, puis évalué sur `Z_TEST_15` plus `n_noise` colonnes de bruit ; le bruit suit une loi normale standard, tiré par `rng = np.random.default_rng(seed)` en deux appels : `rng.normal(size=(len(Z_TRAIN_15), n_noise))` pour l'entraînement, puis `rng.normal(size=(len(Z_TEST_15), n_noise))` pour le test ;
- `distance_ratio_15(p, seed=1315)` : 500 points tirés uniformément dans le cube $[0, 1]^p$ ; pour chacun des 50 premiers, la distance à son plus proche **autre** point divisée par la distance à son point le plus lointain (ton `pairwise_distances`) ; la fonction rend la moyenne de ces 50 rapports ;
- a) `accuracies_15` : pour chaque nombre de colonnes de `NOISE_15`, la moyenne de `noisy_accuracy_15` sur les 10 graines de `SEEDS_15` ;
- b) `first_below_15` : le plus petit nombre de colonnes de `NOISE_15` dont l'accuracy moyenne passe sous 0,9 ;
- c) `ratios_15` : `distance_ratio_15(p)` pour chaque dimension de `P_15`.

La vérification contrôle a) à c), puis trace les deux courbes.

Dans tes notes : les colonnes de bruit ont la même échelle que les mesures standardisées ; pourquoi finissent-elles par décider des voisins ? Que dit c) sur le sens de « plus proche voisin » en grande dimension ? Que ferais-tu sur de vraies données qui ont beaucoup de features ?

In [ ]:
SCALER_15 = StandardScaler().fit(P_TRAIN[MEASURES])
Z_TRAIN_15, Z_TEST_15 = SCALER_15.transform(P_TRAIN[MEASURES]), SCALER_15.transform(P_TEST[MEASURES])
Y_TRAIN_15, Y_TEST_15 = P_TRAIN["species"].to_numpy(), P_TEST["species"].to_numpy()
NOISE_15 = [0, 2, 5, 10, 20, 50, 100, 200]   # numbers of noise columns
SEEDS_15 = range(10)                          # 10 draws of the noise for each number
P_15 = [2, 10, 100, 1000]                     # dimensions of the uniform cube

In [ ]:
def noisy_accuracy_15(n_noise, seed):
    """Test accuracy of YOUR mylearn.neighbors.KNeighborsClassifier(n_neighbors=5), fitted on the 4 standardized
    measures Z_TRAIN_15 plus n_noise columns of noise, then scored on Z_TEST_15 plus n_noise columns of noise. The noise
    is standard normal, drawn with rng = np.random.default_rng(seed) in two calls: rng.normal(size=(len(Z_TRAIN_15),
    n_noise)) for the training part, then rng.normal(size=(len(Z_TEST_15), n_noise)) for the test part."""
    raise NotImplementedError("noisy_accuracy_15")


def distance_ratio_15(p, seed=1315):
    """Draw 500 points uniformly in [0, 1]^p (np.random.default_rng(seed).uniform(size=(500, p))). For each of the
    first 50, the distance to its nearest OTHER point divided by the distance to its farthest point (YOUR
    mylearn.neighbors.pairwise_distances); return the mean of these 50 ratios."""
    raise NotImplementedError("distance_ratio_15")


accuracies_15 = ...   # a) list: for each n_noise of NOISE_15, the mean of noisy_accuracy_15(n_noise, seed) over SEEDS_15
first_below_15 = ...  # b) the smallest n_noise of NOISE_15 whose mean accuracy in a) is below 0.9
ratios_15 = ...       # c) list: distance_ratio_15(p) for each p of P_15

In [ ]:
wb.check("13.15a", accuracies_15)
wb.check("13.15b", first_below_15)
wb.check("13.15c", ratios_15)
with wb.attempt("13.15"):
    if not filled(accuracies_15, ratios_15):
        raise NotImplementedError("accuracies_15, ratios_15")
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(NOISE_15, accuracies_15, marker="o")
    axes[0].set_xscale("symlog", linthresh=2)
    axes[0].set_xlabel("noise columns added to the 4 measures")
    axes[0].set_ylabel("mean test accuracy (k = 5)")
    axes[1].plot(P_15, ratios_15, marker="o")
    axes[1].set_xscale("log")
    axes[1].set_ylim(0, 1)
    axes[1].set_xlabel("dimension p of the cube")
    axes[1].set_ylabel("nearest / farthest distance")
    plt.show()

## Partie B · Les machines à vecteurs de support

Les SVM de scikit-learn (`sklearn.svm.SVC`) : la marge et les vecteurs de support d'un SVM linéaire (13.16), une transformation que tu calcules toi-même pour séparer des cercles par un plan (13.17), puis le noyau RBF réglé par validation croisée (13.18). Le workbook ne te fait pas coder de SVM : l'optimisation de la marge demande des outils (la programmation quadratique) qui dépassent le chapitre. La cellule suivante définit `draw_band`, qui trace la bande d'un SVM en 2D.

In [ ]:
def draw_band(ax, model, X, y, title):
    """The points, the boundary f(x) = 0 and the two edges f(x) = ±1 of a 2D SVC, its support vectors circled."""
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300),
                         np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300))
    f = model.decision_function(np.column_stack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
    ax.contourf(xx, yy, f > 0, alpha=0.15, cmap="coolwarm")
    ax.contour(xx, yy, f, levels=[-1, 0, 1], colors="black", linestyles=["--", "-", "--"], linewidths=1)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=14, edgecolors="white", linewidths=0.4)
    sv = model.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=60, facecolors="none", edgecolors="black", linewidths=0.8)
    ax.set_title(title)

### Ex 13.16 — SVM linéaire : vecteurs de support et paramètre C 📦 ★★ ⏱️ 25 min
**Objectif :** lire la marge et les vecteurs de support d'un SVM linéaire, et voir ce que C change.  
**Prérequis :** ✏️ 13.4 · fiche §13.4 · **Fil rouge :** synthétique · **Parcours :** R, M, C

`X_16` : 200 points de deux classes (0 et 1), deux nuages gaussiens qui se chevauchent un peu. On entraîne `SVC(kernel="linear", C=C)` pour les trois valeurs de `CS_16` ; sa fonction de décision est $f(\mathbf{x}) = \mathbf{w} \cdot \mathbf{x} + b$, et la bande est l'ensemble des points où $-1 < f(\mathbf{x}) < 1$. Calcule :
a) `n_support_16` : la liste des nombres totaux de vecteurs de support, un par valeur de `CS_16` (dans cet ordre) ;
b) `width_16` : pour $C = 1$, la largeur de la bande, la distance entre les droites $f(\mathbf{x}) = -1$ et $f(\mathbf{x}) = 1$ ;
c) `inside_16` : pour $C = 1$, le nombre de points tels que $y_i f(\mathbf{x}_i) < 0{,}999$, avec $y_i = -1$ pour la classe 0 et $+1$ pour la classe 1 (les points dans la bande ou du mauvais côté ; le seuil 0,999 tolère l'arrondi des points posés sur un bord).

La vérification contrôle a) à c), puis trace les trois bandes, vecteurs de support entourés.

Dans tes notes : comment la largeur de la bande et le nombre de vecteurs de support varient-ils avec C, et pourquoi ? Réentraîne le SVM ($C = 1$) sur ses seuls vecteurs de support : que deviennent $\mathbf{w}$ et $b$, et qu'est-ce que cela dit de ce que le SVM « retient » des données ?

In [ ]:
X_16, Y_16 = wb.synth.make_blobs(n=200, centers=[[0.0, 0.0], [2.5, 2.0]], std=0.9, seed=1316)   # two classes, 0 and 1
CS_16 = [0.01, 1, 100]

In [ ]:
n_support_16 = ...   # a) list: the total number of support vectors of SVC(kernel="linear", C=C) fitted on X_16, Y_16, for each C of CS_16
width_16 = ...       # b) for C = 1: the width of the band between the two edges f(x) = -1 and f(x) = +1
inside_16 = ...      # c) for C = 1: the number of points with y_i * f(x_i) < 0.999 (y_i = -1 for class 0, +1 for class 1)

In [ ]:
wb.check("13.16a", n_support_16)
wb.check("13.16b", width_16)
wb.check("13.16c", inside_16)
with wb.attempt("13.16"):
    if not filled(n_support_16, width_16, inside_16):
        raise NotImplementedError("n_support_16, width_16, inside_16")
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, C in zip(axes, CS_16):
        model = SVC(kernel="linear", C=C).fit(X_16, Y_16)
        draw_band(ax, model, X_16, Y_16, f"C = {C}")
    plt.show()

### Ex 13.17 — Relever les cercles en 3D pour les séparer par un plan 🎨 ★★ ⏱️ 30 min
**Objectif :** voir qu'une transformation des features rend linéaire ce qui ne l'était pas, et le dessiner.  
**Prérequis :** Ex 13.16 · ✏️ 13.5 · fiche §13.4 (astuce du noyau) · **Fil rouge :** synthétique · **Parcours :** M, C

`X_17` : un anneau (classe 0) autour d'un disque (classe 1), deux cercles bruités de rayons 1 et 0,4 ; 70 % pour l'entraînement, 30 % pour le test. Aucune droite ne les sépare ; on les **relève** en ajoutant une troisième coordonnée, $x_3 = x_1^2 + x_2^2$, le carré de la distance au centre. Écris :
- `lift_17(X)` : le tableau `(n, 3)` des deux coordonnées, puis de $x_1^2 + x_2^2$ ;
- `draw_17(ax, X3, y, model)` : sur un axe 3D, les points relevés colorés par classe, et le plan $\mathbf{w} \cdot \mathbf{z} + b = 0$ du SVM linéaire `model` (résous-le en $z_3$ sur une grille de $(z_1, z_2)$, et trace-le avec `ax.plot_surface`) ;
- a) `acc_2d_17` : l'accuracy de test de `SVC(kernel="linear", C=1)` entraîné sur les deux coordonnées ;
- b) `acc_3d_17` : la même chose sur les trois coordonnées relevées ;
- c) `radius_17` : la frontière de b), revenue dans le plan des données, est un cercle : son rayon (garde tous les chiffres).

La vérification contrôle a) à c), essaie `lift_17` sur deux points, puis dessine la frontière vue de dessus et ta figure 3D.

Dans tes notes : pourquoi le rayon trouvé tombe-t-il entre 0,4 et 1 ? Écris le produit scalaire $\varphi(\mathbf{x}) \cdot \varphi(\mathbf{z})$ de ta transformation en fonction de $\mathbf{x}$ et $\mathbf{z}$ seulement : quel noyau as-tu, sans le savoir, utilisé ? Qu'aurait fait un noyau RBF sur ces données ?

In [ ]:
X_17, Y_17 = wb.synth.make_circles(n=400, noise=0.1, factor=0.4, seed=1317)   # a ring (class 0) around a disc (class 1)
X_TRAIN_17, X_TEST_17, Y_TRAIN_17, Y_TEST_17 = train_test_split(X_17, Y_17, test_size=0.3, random_state=1317,
                                                                stratify=Y_17)

In [ ]:
def lift_17(X):
    """(n, 2) -> (n, 3): the two coordinates, then x1**2 + x2**2."""
    raise NotImplementedError("lift_17")


def draw_17(ax, X3, y, model):
    """On the 3D axis `ax`: the lifted points X3 (n, 3) coloured by class y, and the plane w . z + b = 0 of `model`, a
    linear SVC fitted on lifted points (solve the plane for its third coordinate on a grid of (x1, x2))."""
    raise NotImplementedError("draw_17")


acc_2d_17 = ...   # a) test accuracy of SVC(kernel="linear", C=1) fitted on the 2 coordinates of X_TRAIN_17
acc_3d_17 = ...   # b) the same, fitted and scored on the lifted coordinates (lift_17)
radius_17 = ...   # c) the radius of the circle where the boundary of b) cuts the surface x3 = x1**2 + x2**2

In [ ]:
wb.check("13.17a", acc_2d_17)
wb.check("13.17b", acc_3d_17)
wb.check("13.17c", radius_17)
with wb.attempt("13.17"):
    probe_17 = lift_17(np.array([[1.0, 2.0], [0.0, -3.0]]))
    if returned("13.17", "lift_17", probe_17):
        verdict("13.17", np.shape(probe_17) == (2, 3) and np.allclose(probe_17, [[1, 2, 5], [0, -3, 9]]),
                "lift_17 ajoute bien x1² + x2² en troisième colonne.",
                f"lift_17([[1, 2], [0, -3]]) rend {np.round(np.asarray(probe_17, dtype=float), 3).tolist()} au lieu de "
                "[[1, 2, 5], [0, -3, 9]] : les deux coordonnées, puis la somme de leurs carrés.")
        model_17 = SVC(kernel="linear", C=1).fit(lift_17(X_TRAIN_17), Y_TRAIN_17)
        fig = plt.figure(figsize=(13, 5))
        ax_17 = fig.add_subplot(1, 2, 1)
        wb.plot.plot_decision_boundary(lambda points: model_17.predict(lift_17(points)), X_TEST_17, Y_TEST_17,
                                       ax=ax_17, title="the plane, seen from above (test points)")
        draw_17(fig.add_subplot(1, 2, 2, projection="3d"), lift_17(X_TRAIN_17), Y_TRAIN_17, model_17)
        plt.show()

### Ex 13.18 — Noyau RBF : régler C et gamma par validation croisée 📦 ★★★ ⏱️ 35 min
**Objectif :** régler ensemble C et $\gamma$ sur une grille logarithmique, et reconnaître un $\gamma$ trop grand.  
**Prérequis :** Ex 13.16 · ch. 8 (validation croisée) · fiche §13.4 · **Fil rouge :** synthétique · **Parcours :** C

`X_18` : deux spirales entrelacées et bruitées (400 points, 70 % pour l'entraînement). `GridSearchCV(estimator, param_grid, cv=...)` de scikit-learn (le ch. 15 y revient en détail) essaie toutes les combinaisons du dictionnaire `param_grid` en validation croisée, puis réentraîne la meilleure sur toute la partie d'entraînement. Après `fit`, son attribut `best_params_` est le dictionnaire de la combinaison retenue, et sa méthode `score(X, y)` note le modèle réentraîné. Avec `GridSearchCV` et `SVC(kernel="rbf")`, essaie les 25 combinaisons de `GRID_18` en validation croisée à 5 folds (`CV_18`), sur la partie d'entraînement. Calcule :
a) `best_18` : la liste `[C, gamma]` de la meilleure combinaison ;
b) `best_cv_18` : son accuracy moyenne en validation croisée ;
c) `test_18` : l'accuracy de test du modèle réentraîné sur toute la partie d'entraînement avec cette combinaison ;
d) `overfit_18` : la liste `[accuracy d'entraînement, accuracy de test]` de `SVC(C=1000, gamma=1000)` entraîné sur la partie d'entraînement.

La vérification contrôle a) à d), puis trace la carte des accuracies de validation croisée et deux frontières.

Dans tes notes : décris la carte : où sont les bonnes combinaisons ? Que fait un noyau de $\gamma = 1\,000$ autour de chaque vecteur de support, et pourquoi C n'y change presque rien ? Pourquoi la grille est-elle logarithmique ?

In [ ]:
X_18, Y_18 = wb.synth.make_spirals(n=400, noise=0.8, seed=1318)   # two intertwined spirals
X_TRAIN_18, X_TEST_18, Y_TRAIN_18, Y_TEST_18 = train_test_split(X_18, Y_18, test_size=0.3, random_state=1318,
                                                                stratify=Y_18)
GRID_18 = {"C": [0.1, 1, 10, 100, 1000], "gamma": [0.1, 1, 10, 100, 1000]}
CV_18 = StratifiedKFold(n_splits=5, shuffle=True, random_state=1318)

In [ ]:
search_18 = ...    # GridSearchCV(SVC(kernel="rbf"), GRID_18, cv=CV_18), fitted on X_TRAIN_18, Y_TRAIN_18
best_18 = ...      # a) [C, gamma] of the best combination
best_cv_18 = ...   # b) its mean accuracy in cross-validation
test_18 = ...      # c) the test accuracy of search_18 (refitted on the whole training part)
overfit_18 = ...   # d) [training accuracy, test accuracy] of SVC(C=1000, gamma=1000) fitted on the training part

In [ ]:
wb.check("13.18a", best_18)
wb.check("13.18b", best_cv_18)
wb.check("13.18c", test_18)
wb.check("13.18d", overfit_18)
with wb.attempt("13.18"):
    if not filled(search_18) or not hasattr(search_18, "cv_results_"):
        raise NotImplementedError("search_18")
    scores_18 = search_18.cv_results_["mean_test_score"].reshape(len(GRID_18["C"]), len(GRID_18["gamma"]))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    image = axes[0].imshow(scores_18, cmap="viridis", vmin=0.5, vmax=1)
    axes[0].set_xticks(range(5), GRID_18["gamma"])
    axes[0].set_yticks(range(5), GRID_18["C"])
    axes[0].set_xlabel("gamma")
    axes[0].set_ylabel("C")
    axes[0].set_title("mean accuracy in cross-validation")
    axes[0].grid(False)
    for i in range(5):
        for j in range(5):
            axes[0].text(j, i, f"{scores_18[i, j]:.2f}", ha="center", va="center", color="white", fontsize=8)
    fig.colorbar(image, ax=axes[0])
    wb.plot.plot_decision_boundary(search_18.best_estimator_, X_TEST_18, Y_TEST_18, ax=axes[1],
                                   title=f"best: C = {search_18.best_params_['C']}, gamma = {search_18.best_params_['gamma']}")
    wb.plot.plot_decision_boundary(SVC(C=1000, gamma=1000).fit(X_TRAIN_18, Y_TRAIN_18), X_TEST_18, Y_TEST_18,
                                   ax=axes[2], title="C = 1000, gamma = 1000")
    plt.show()

## Partie C · Les arbres de décision

D'abord les arbres de scikit-learn : leur instabilité (13.19) et une colonne qui triche (13.21), puis l'élagage (13.23). En parallèle, tu construis `mylearn/tree.py` pièce par pièce : les impuretés et le gain d'un split (13.20), la recherche du meilleur split (13.22), l'arbre de classification (13.24), ses options pour les ensembles du ch. 14 (13.25), puis l'arbre de régression (13.26). Ces cinq exercices 🔨 forment un tout : ton arbre servira **tel quel** au ch. 14. La cellule suivante définit `tree_rules`, qui affiche les règles d'un arbre de ta librairie, et `as_float32`, qui arrondit des données comme le fait scikit-learn (ses arbres travaillent en `float32`), pour comparer les deux librairies à l'identique.

In [ ]:
from sklearn.tree import export_text, plot_tree


def tree_rules(node, feature_names, class_names=None, indent=""):
    """The rules of a mylearn tree (a Node and its children), one line per test, like export_text."""
    if node.left is None:
        value = np.ravel(np.asarray(node.value, dtype=float))
        what = f"class: {class_names[int(np.argmax(value))]}" if class_names is not None else f"value: {value[0]:.3f}"
        return [f"{indent}|--- {what}  ({node.n_samples} samples)"]
    name = feature_names[node.feature]
    return ([f"{indent}|--- {name} <= {node.threshold:.2f}"]
            + tree_rules(node.left, feature_names, class_names, indent + "|   ")
            + [f"{indent}|--- {name} >  {node.threshold:.2f}"]
            + tree_rules(node.right, feature_names, class_names, indent + "|   "))


def as_float32(X):
    """X rounded to float32, then back to float64: scikit-learn's trees work in float32, so this gives both libraries
    exactly the same values (and the same thresholds)."""
    return np.asarray(X, dtype=np.float32).astype(np.float64)

### Ex 13.19 — Instabilité et profondeur d'un arbre 🔬 ★★ ⏱️ 30 min
**Objectif :** mesurer combien un arbre change d'un échantillon à l'autre, et ce que la profondeur y fait.  
**Prérequis :** Ex 13.12 · ch. 9 (overfitting, biais et variance) · fiche §13.5.1 et §13.5.3 · **Fil rouge :** synthétique · **Parcours :** R, C

`X_19` : 600 points de deux demi-lunes **très** bruitées. `DRAWS_19` contient 20 tirages de 70 % des points (420 indices chacun). Écris :
- `trees_19(max_depth)` : la liste des 20 arbres `DecisionTreeClassifier(max_depth=max_depth, random_state=0)`, chacun entraîné sur un tirage ;
- `disagreement_19(trees)` : la moyenne, sur toutes les paires d'arbres de la liste, de la part des points de `GRID_19` (30 000 points du plan) où les deux arbres prédisent des classes différentes ;
- a) `leaves_19` : le nombre moyen de feuilles des 20 arbres sans limite ;
- b) `disagree_19` : la liste `[désaccord des arbres sans limite, désaccord des arbres de profondeur 3]` ;
- c) `root_range_19` : l'écart entre le plus grand et le plus petit seuil de la racine des 20 arbres sans limite (`tree_.threshold[0]`) ;
- d) `best_depth_19` : la profondeur de `DEPTHS_19` (de 1 à 12) qui a la meilleure accuracy moyenne en validation croisée à 5 folds (`CV_19`) sur les 600 points.

La vérification contrôle a) à d), puis trace quatre arbres et les courbes d'accuracy (entraînement et validation croisée) selon la profondeur.

Dans tes notes : les racines coupent-elles toujours la même feature ? Pourquoi le seuil varie-t-il autant, et pourquoi toute la suite de l'arbre en dépend-elle ? Pourquoi les arbres de profondeur 3 sont-ils plus stables ? Que disent les deux courbes sur le biais et la variance ?

In [ ]:
X_19, Y_19 = wb.synth.make_moons(n=600, noise=0.35, seed=1319)
DRAWS_19 = [np.random.default_rng(1319 + r).choice(600, 420, replace=False) for r in range(20)]   # 20 draws of 70 %
grid1_19, grid2_19 = np.meshgrid(np.linspace(-1.5, 2.5, 200), np.linspace(-1.2, 1.7, 150))
GRID_19 = np.column_stack([grid1_19.ravel(), grid2_19.ravel()])   # 30,000 points of the plane
DEPTHS_19 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
CV_19 = StratifiedKFold(n_splits=5, shuffle=True, random_state=1319)

In [ ]:
def trees_19(max_depth):
    """The list of 20 trees DecisionTreeClassifier(max_depth=max_depth, random_state=0), one fitted on each draw of
    DRAWS_19 (the rows X_19[draw], Y_19[draw])."""
    raise NotImplementedError("trees_19")


def disagreement_19(trees):
    """Mean, over all the pairs of trees of the list, of the share of the points of GRID_19 where the two trees
    predict different classes."""
    raise NotImplementedError("disagreement_19")


leaves_19 = ...       # a) the mean number of leaves of the 20 trees of trees_19(None)
disagree_19 = ...     # b) [disagreement_19(trees_19(None)), disagreement_19(trees_19(3))]
root_range_19 = ...   # c) the largest minus the smallest root threshold (tree_.threshold[0]) of the trees of trees_19(None)
best_depth_19 = ...   # d) the max_depth of DEPTHS_19 with the best mean 5-fold accuracy (CV_19) on X_19, Y_19

In [ ]:
wb.check("13.19a", leaves_19)
wb.check("13.19b", disagree_19)
wb.check("13.19c", root_range_19)
wb.check("13.19d", best_depth_19)
with wb.attempt("13.19"):
    if not filled(leaves_19, best_depth_19):
        raise NotImplementedError("leaves_19, best_depth_19")
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    for ax, (depth, r) in zip(axes, [(None, 0), (None, 1), (3, 0), (3, 1)]):
        tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_19[DRAWS_19[r]], Y_19[DRAWS_19[r]])
        wb.plot.plot_decision_boundary(tree, X_19[DRAWS_19[r]], Y_19[DRAWS_19[r]], ax=ax,
                                       title=f"draw {r}, max_depth={depth}: {tree.get_n_leaves()} leaves")
    plt.show()
    train_19 = [DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_19, Y_19).score(X_19, Y_19) for d in DEPTHS_19]
    cv_19 = [cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=0), X_19, Y_19, cv=CV_19).mean()
             for d in DEPTHS_19]
    labels_19 = [str(d) for d in DEPTHS_19]
    plt.plot(labels_19, train_19, marker="o", label="training")
    plt.plot(labels_19, cv_19, marker="o", label="5-fold cross-validation")
    plt.xlabel("max_depth")
    plt.ylabel("accuracy")
    plt.legend()
    plt.show()

### Ex 13.20 — Coder les impuretés et le gain d'un split 🔨 ★★ ⏱️ 25 min
**Objectif :** écrire les trois impuretés et le gain pondéré d'un split, avec des poids d'exemples.  
**Prérequis :** ✏️ 13.2 et 13.3 · ch. 6 (entropie en bits) · fiche §13.5.2 · **mylearn :** `tree.py` · **Parcours :** M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/tree.py` (créé par `python tools/start_chapter.py 13`), lis la docstring de chaque fonction et de chaque méthode, remplace les `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy seulement : scikit-learn et SciPy sont les **oracles** des tests. La cellule de vérification recharge ta librairie, compare quelques résultats à scikit-learn, puis lance les tests de l'exercice ; `python -m pytest tests/test_ch13_tree.py -q` les lance tous dans un terminal.

Écris `gini_impurity`, `entropy_impurity`, `variance_impurity` et `split_gain`. **Contrat** (les docstrings, plus ces précisions que vérifient les tests) :
- les trois impuretés reçoivent `y` (une liste ou un tableau 1-D, des labels quelconques pour les deux premières, des nombres pour la variance) et `sample_weight` (`None` : tous les poids valent 1) et rendent un `float` Python ; les proportions et la moyenne sont **pondérées** : un poids 2 compte comme un exemple répété deux fois, une classe de poids nul ne compte pas (et ne donne pas de `nan` dans l'entropie) ;
- l'entropie est en **bits** ($\log_2$, avec $0 \log 0 = 0$) ; la variance est la moyenne pondérée des carrés des écarts à la moyenne pondérée (ddof = 0) ;
- `split_gain(y, left_mask, criterion="gini", sample_weight=None)` rend $I(\text{parent}) - \frac{w_G}{w} I(G) - \frac{w_D}{w} I(D)$, avec les poids totaux $w_G$, $w_D$ et $w$ (les nombres d'exemples quand les poids valent 1), pour `criterion` dans `"gini"`, `"entropy"`, `"squared_error"` ; si un enfant est vide, le gain vaut `0.0` ;
- erreurs (`ValueError`) : `y` vide ; des poids négatifs, de mauvaise forme ou de somme nulle ; un critère inconnu ; des formes différentes pour `y`, `left_mask` et `sample_weight`.

La vérification essaie les exemples des docstrings, compare l'impureté de la racine et le gain du premier split d'un arbre scikit-learn sur les manchots, puis lance les tests.

Dans tes notes : pourquoi pondérer chaque enfant par sa part des exemples ? Que vaudrait le « gain » d'un split qui envoie un seul exemple à gauche, si l'on oubliait la pondération ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="13")   # reload your saved file
with checked("13.20"):
    cases_20 = [("gini_impurity([0, 0, 1, 1])", lambda: mylearn.tree.gini_impurity([0, 0, 1, 1]), 0.5),
                ('gini_impurity(["a", "a", "a", "b"])', lambda: mylearn.tree.gini_impurity(["a", "a", "a", "b"]), 0.375),
                ("gini_impurity([0, 1], sample_weight=[3.0, 1.0])",
                 lambda: mylearn.tree.gini_impurity([0, 1], sample_weight=[3.0, 1.0]), 0.375),
                ("entropy_impurity([0, 0, 0, 1])", lambda: mylearn.tree.entropy_impurity([0, 0, 0, 1]), 0.811278),
                ("variance_impurity([1.0, 2.0, 3.0, 4.0])", lambda: mylearn.tree.variance_impurity([1.0, 2.0, 3.0, 4.0]), 1.25),
                ('split_gain([0, 0, 1, 1], [True, True, False, False], criterion="entropy")',
                 lambda: mylearn.tree.split_gain([0, 0, 1, 1], [True, True, False, False], criterion="entropy"), 1.0)]
    for text_20, call_20, want_20 in cases_20:
        try:
            got_20 = call_20()
        except NotImplementedError:
            print(f"⏳ {text_20} : pas encore écrite")
            continue
        ok_20 = isinstance(got_20, (int, float, np.floating)) and np.isclose(got_20, want_20, atol=1e-6)
        print(("✅ " if ok_20 else "❌ ") + f"{text_20} = {got_20!r} (expected {want_20})")
with checked("13.20"):
    X_20, y_20 = P_TRAIN[MEASURES].to_numpy(), P_TRAIN["species"].to_numpy()
    for criterion, impurity_20 in [("gini", mylearn.tree.gini_impurity), ("entropy", mylearn.tree.entropy_impurity)]:
        t_20 = DecisionTreeClassifier(criterion=criterion, max_depth=1, random_state=0).fit(X_20, y_20).tree_
        left_20, right_20 = t_20.children_left[0], t_20.children_right[0]
        sk_gain_20 = t_20.impurity[0] - (t_20.n_node_samples[left_20] * t_20.impurity[left_20]
                                         + t_20.n_node_samples[right_20] * t_20.impurity[right_20]) / t_20.n_node_samples[0]
        mask_20 = as_float32(X_20)[:, t_20.feature[0]] <= t_20.threshold[0]
        root_20 = impurity_20(y_20)
        gain_20 = mylearn.tree.split_gain(y_20, mask_20, criterion=criterion)
        if not (returned("13.20", f"{criterion}_impurity", root_20) and returned("13.20", "split_gain", gain_20)):
            continue
        if not all(isinstance(value, (int, float, np.floating)) for value in (root_20, gain_20)):
            print(f"❌ Ex 13.20 : {criterion}_impurity et split_gain doivent rendre un nombre (un float), pas "
                  f"{type(root_20).__name__} et {type(gain_20).__name__}.")
        else:
            verdict("13.20", np.isclose(root_20, t_20.impurity[0]) and np.isclose(gain_20, sk_gain_20),
                    f"{criterion} : impureté de la racine {fr(root_20)} et gain du premier split de scikit-learn "
                    f"{fr(gain_20)}, comme scikit-learn.",
                    f"{criterion} : scikit-learn trouve une impureté de {fr(t_20.impurity[0])} à la racine et un gain "
                    f"de {fr(sk_gain_20)} pour son premier split ; les tests ci-dessous disent où est l'écart.")
if written(lambda: mylearn.tree.gini_impurity([0, 1])):
    run_mylearn_tests("tree", keyword="test_gini_impurity_ or test_entropy_impurity_ or test_variance_impurity_ or test_split_gain_")
else:
    print("⏳ Ex 13.20 : les tests attendent ton gini_impurity.")

### Ex 13.21 — L'arbre parfait qui triche : une colonne identifiant 🐛 ★★ ⏱️ 20 min
**Objectif :** repérer qu'un modèle trop beau s'appuie sur une colonne qui n'apprend rien du phénomène.  
**Prérequis :** Ex 13.19 · ch. 12 (lecture des fichiers, fuites) · fiche §13.5.3 (pièges : les `feature_importances_`) · **Fil rouge :** Penguins · **Parcours :** C

Un collègue enregistre les 333 manchots dans un fichier CSV, le relit, et entraîne un arbre sur « toutes les colonnes sauf l'espèce ». Son arbre est **parfait** sur le test : 100 %. La saison suivante, les manchots du test sont remesurés et ajoutés au fichier : `NEXT_21`. Mène l'enquête, sans modifier les données du collègue :
a) `top_feature_21` : le nom de la feature la plus importante de `TREE_21` (`feature_importances_`) ;
b) `next_accuracy_21` : l'accuracy de `TREE_21` sur `NEXT_21` ;
c) `clean_accuracy_21` : l'accuracy de test (`TEST_21`) d'un `DecisionTreeClassifier(random_state=0)` entraîné sur `TRAIN_21` **sans** la colonne fautive ;
d) `clean_leaves_21` : le nombre de feuilles de l'arbre de c).

Dans tes notes : d'où vient la colonne fautive, et pourquoi suffit-elle à classer parfaitement ? Combien de feuilles lui faut-il, et pourquoi exactement ce nombre ? Qu'arrive-t-il aux manchots de la saison suivante ? Comment le collègue aurait-il dû écrire et relire son fichier, et quel contrôle aurait révélé le problème avant le test ?

In [ ]:
# A colleague saves the 333 penguins in a CSV file (index included, the default of to_csv), then reads it back
CSV_21 = PENGUINS[MEASURES + ["species"]].to_csv()
DATA_21 = pd.read_csv(io.StringIO(CSV_21))
FEATURES_21 = [column for column in DATA_21.columns if column != "species"]   # "every column but the target"
TRAIN_21, TEST_21 = train_test_split(DATA_21, test_size=0.3, random_state=1321, stratify=DATA_21["species"])
TREE_21 = DecisionTreeClassifier(random_state=0).fit(TRAIN_21[FEATURES_21], TRAIN_21["species"])
print(f"the colleague's tree: test accuracy {TREE_21.score(TEST_21[FEATURES_21], TEST_21['species']):.3f}")
# Next season: the test penguins are measured again, appended to the file, which is saved and read back
NEXT_FILE_21 = pd.concat([PENGUINS[MEASURES + ["species"]], PENGUINS.loc[TEST_21.index, MEASURES + ["species"]]],
                         ignore_index=True).to_csv()
NEXT_21 = pd.read_csv(io.StringIO(NEXT_FILE_21)).iloc[len(PENGUINS):]
print(len(NEXT_21), "penguins next season")

In [ ]:
top_feature_21 = ...      # a) the name of the feature of FEATURES_21 with the largest importance in TREE_21
next_accuracy_21 = ...    # b) the accuracy of TREE_21 on NEXT_21
clean_accuracy_21 = ...   # c) the test accuracy (TEST_21) of DecisionTreeClassifier(random_state=0) fitted on TRAIN_21
                          #    without the guilty column
clean_leaves_21 = ...     # d) the number of leaves of the tree of c)

In [ ]:
wb.check("13.21a", top_feature_21)
wb.check("13.21b", next_accuracy_21)
wb.check("13.21c", clean_accuracy_21)
wb.check("13.21d", clean_leaves_21)

### Ex 13.22 — Coder la recherche du meilleur split 🔨 ★★★ ⏱️ 45 min
**Objectif :** écrire la recherche gloutonne d'un split, avec les conventions qui la rendent identique à scikit-learn.  
**Prérequis :** Ex 13.20 · ✏️ 13.3 · fiche §13.5.1 et §13.5.2 · **Fil rouge :** Penguins · **mylearn :** `tree.py` · **Parcours :** M, C

> **mylearn** : dans `mon_travail/mylearn/tree.py`, mêmes règles qu'en 13.20 (NumPy seulement, scikit-learn pour oracle). Enregistre, puis relance la cellule de vérification.

Écris `best_split`. **Contrat** (la docstring, plus ces précisions que vérifient les tests) :
- `best_split(X, y, criterion="gini", sample_weight=None, feature_indices=None, min_samples_leaf=1, splitter="best", rng=None)` rend `(feature, threshold, gain)` (un `int` et deux `float` Python) ou `None` ; les exemples tels que `X[:, feature] <= threshold` vont à gauche ;
- `splitter="best"` : les seuils candidats d'une feature sont les **milieux** de deux valeurs distinctes consécutives (triées) ; `splitter="random"` (pour les ExtraTrees du ch. 14) : un seul seuil par feature, tiré par `rng.uniform(min, max)` des valeurs du nœud, dans l'ordre de `feature_indices` ; une feature constante est sautée (aucun tirage) ; `rng=None` : `np.random.default_rng()` ;
- `feature_indices` : les features à examiner (`None` : toutes) ; la feature rendue est son indice **dans `X`** ;
- un candidat n'est valide que si chaque enfant reçoit au moins `min_samples_leaf` exemples (comptés sans les poids) ; son gain est celui de `split_gain`, avec les poids ;
- le meilleur gain l'emporte ; les égalités se jugent **à l'arrondi près** (un candidat ne remplace le meilleur que si son gain le dépasse de plus de $10^{-12}$ environ) et vont à la plus petite feature, puis au plus petit seuil, quel que soit l'ordre de `feature_indices` ;
- `None` si aucun candidat valide n'a un gain strictement positif (au-delà de l'arrondi) : le nœud deviendra une feuille ; la fonction ne modifie pas ses arguments ; `ValueError` pour un critère ou un `splitter` inconnu, ou des formes incohérentes.

La vérification essaie l'exemple de la docstring, compare ton premier split des manchots à celui de scikit-learn (Gini et entropie), puis lance les tests.

Dans tes notes : combien de seuils candidats examines-tu pour la longueur des nageoires des 233 manchots ? Pourquoi juger les égalités à l'arrondi près, et pourquoi la plus petite feature ? Pourquoi un gain nul doit-il arrêter ta recherche (le « ou exclusif » de la fiche) ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="13")   # reload your saved file
with checked("13.22"):
    doc_22 = mylearn.tree.best_split(np.array([[1.0, 5.0], [2.0, 3.0], [3.0, 8.0], [4.0, 1.0]]), np.array([0, 0, 1, 1]))
    if returned("13.22", "best_split", doc_22):
        print("docstring example:", doc_22, "(expected (0, 2.5, 0.5))")
with checked("13.22"):
    X_22, y_22 = as_float32(P_TRAIN[MEASURES]), P_TRAIN["species"].to_numpy()
    for criterion in ["gini", "entropy"]:
        t_22 = DecisionTreeClassifier(criterion=criterion, max_depth=1, random_state=0).fit(X_22, y_22).tree_
        left_22, right_22 = t_22.children_left[0], t_22.children_right[0]
        sk_gain_22 = t_22.impurity[0] - (t_22.n_node_samples[left_22] * t_22.impurity[left_22]
                                         + t_22.n_node_samples[right_22] * t_22.impurity[right_22]) / t_22.n_node_samples[0]
        mine_22 = mylearn.tree.best_split(X_22, y_22, criterion=criterion)
        ok_22 = (isinstance(mine_22, tuple) and len(mine_22) == 3 and mine_22[0] == t_22.feature[0]
                 and np.isclose(mine_22[1], t_22.threshold[0]) and np.isclose(mine_22[2], sk_gain_22))
        verdict("13.22", ok_22,
                f"{criterion} : {MEASURES[t_22.feature[0]]} <= {fr(t_22.threshold[0], 2)}, gain {fr(sk_gain_22)}, comme "
                "la racine de scikit-learn.",
                f"{criterion} : best_split rend {mine_22} ; scikit-learn coupe la feature {t_22.feature[0]} "
                f"({MEASURES[t_22.feature[0]]}) en {t_22.threshold[0]:.4f}, avec un gain de {sk_gain_22:.6f}.")
if written(lambda: mylearn.tree.best_split(np.array([[0.0], [1.0]]), np.array([0, 1]))):
    run_mylearn_tests("tree", keyword="test_best_split_")
else:
    print("⏳ Ex 13.22 : les tests attendent ton best_split.")

### Ex 13.23 — Arbres scikit-learn : plot_tree, export_text et élagage ccp_alpha 📦 ★★★ ⏱️ 35 min
**Objectif :** élaguer un arbre par validation croisée, puis le lire comme une suite de règles.  
**Prérequis :** Ex 13.19 · fiche §13.5.3 (élagage par coût-complexité) et §13.7 · **Fil rouge :** Penguins · **Parcours :** R, C

Les 333 manchots, découpés en 70 % pour l'entraînement et 30 % pour le test, avec les 4 mesures. Calcule :
- `full_23` : un `DecisionTreeClassifier(random_state=0)` sans limite, entraîné sur la partie d'entraînement ;
- a) `size_23` : la liste `[profondeur, nombre de feuilles]` de `full_23` ;
- `alphas_23` : les valeurs de `ccp_alpha` que rend `full_23.cost_complexity_pruning_path(...)`, **sans la dernière** (celle qui réduit l'arbre à sa racine) ;
- b) `best_alpha_23` : la valeur de `alphas_23` qui donne la meilleure accuracy moyenne en validation croisée à 5 folds (`CV_23`) ; à égalité (`np.isclose`), la **plus grande** ;
- `pruned_23` : l'arbre élagué avec cette valeur, entraîné sur toute la partie d'entraînement ;
- c) `pruned_size_23` : la liste `[profondeur, nombre de feuilles]` de `pruned_23` ;
- d) `test_23` : la liste `[accuracy de test de full_23, accuracy de test de pruned_23]`.

La vérification contrôle a) à d), puis dessine l'arbre élagué (`plot_tree`) et affiche ses règles (`export_text`).

Dans tes notes : pourquoi retirer la dernière valeur, et pourquoi préférer la plus grande en cas d'égalité ? Écris en une phrase française la règle qui reconnaît un Gentoo. L'arbre élagué a-t-il « perdu » sur le test ? Que vaut une différence d'un manchot sur 100 ?

In [ ]:
TRAIN_23, TEST_23 = train_test_split(PENGUINS, test_size=0.3, random_state=1323, stratify=PENGUINS["species"])
X_TRAIN_23, Y_TRAIN_23 = TRAIN_23[MEASURES].to_numpy(), TRAIN_23["species"].to_numpy()
X_TEST_23, Y_TEST_23 = TEST_23[MEASURES].to_numpy(), TEST_23["species"].to_numpy()
CV_23 = StratifiedKFold(n_splits=5, shuffle=True, random_state=1323)
print(len(TRAIN_23), "training penguins,", len(TEST_23), "test penguins")

In [ ]:
full_23 = ...         # DecisionTreeClassifier(random_state=0) fitted on X_TRAIN_23, Y_TRAIN_23
size_23 = ...         # a) [depth, number of leaves] of full_23
alphas_23 = ...       # the ccp_alphas of full_23.cost_complexity_pruning_path(X_TRAIN_23, Y_TRAIN_23), without the last
best_alpha_23 = ...   # b) the alpha of alphas_23 with the best mean 5-fold accuracy (CV_23) of
                      #    DecisionTreeClassifier(ccp_alpha=alpha, random_state=0); the LARGEST one if tied (np.isclose)
pruned_23 = ...       # DecisionTreeClassifier(ccp_alpha=best_alpha_23, random_state=0) fitted on the training part
pruned_size_23 = ...  # c) [depth, number of leaves] of pruned_23
test_23 = ...         # d) [test accuracy of full_23, test accuracy of pruned_23]

In [ ]:
wb.check("13.23a", size_23)
wb.check("13.23b", best_alpha_23)
wb.check("13.23c", pruned_size_23)
wb.check("13.23d", test_23)
with wb.attempt("13.23"):
    if not filled(pruned_23) or not hasattr(pruned_23, "tree_"):
        raise NotImplementedError("pruned_23")
    plt.figure(figsize=(12, 6))
    plot_tree(pruned_23, feature_names=MEASURES, class_names=list(pruned_23.classes_), filled=True, fontsize=9)
    plt.show()
    print(export_text(pruned_23, feature_names=MEASURES))

### Ex 13.24 — Coder un arbre de décision récursif 🔨 ★★★★ ⏱️ 150 min
**Objectif :** construire un arbre CART complet, identique nœud par nœud à celui de scikit-learn.  
**Prérequis :** Ex 13.22 · 0A (fonctions récursives, `@dataclass`) · fiche §13.5 et §13.5.1 (pseudo-code) · **Fil rouge :** Penguins · **mylearn :** `tree.py` · **Parcours :** M, C

> **mylearn** : dans `mon_travail/mylearn/tree.py`, mêmes règles qu'en 13.20. Enregistre, puis relance la cellule de vérification. La classe `Node` est fournie complète.

Écris la classe `DecisionTreeClassifier` (sauf les options du 13.25 : ici, `sample_weight=None`, `max_features=None` et `splitter="best"` suffisent). **Contrat** (la docstring, plus ces précisions que vérifient les tests) :
- `__init__` range ses sept arguments tels quels ; `fit(X, y, sample_weight=None)` vérifie, apprend `classes_` (les labels distincts triés), `n_classes_`, `n_features_in_` et `root_`, puis renvoie `self` ; un second `fit` oublie le premier ; `fit` et `predict` ne modifient pas leurs arguments, et acceptent des listes ;
- un nœud devient une **feuille** s'il est pur, si la profondeur `max_depth` est atteinte, s'il a **moins** de `min_samples_split` exemples, ou si `best_split` ne trouve rien (ta fonction du 13.22, avec `min_samples_leaf`) ; sinon il est coupé, et ses deux enfants se construisent de la même façon ;
- chaque nœud est un `Node` : `feature` et `threshold` (`None` pour une feuille), `left` et `right`, `value` (les proportions des classes dans le nœud, alignées sur `classes_`), `n_samples` (le nombre d'exemples qui l'atteignent) et `impurity` (son impureté pour le critère de l'arbre) ;
- `predict_proba(X)` rend la `value` de la feuille atteinte par chaque exemple ; `predict(X)` la classe de plus grande proportion (à égalité, la première de `classes_`) ; `get_depth()` (0 si la racine est une feuille) ; `get_n_leaves()` ; `score(X, y)` l'accuracy, un `float` Python ;
- erreurs : `ValueError` pour un critère autre que `"gini"` ou `"entropy"`, `max_depth < 1` (`None` est permis), `min_samples_split < 2`, `min_samples_leaf < 1`, un `X` qui n'est pas 2-D ou contient `NaN`, `len(X) != len(y)`, ou un mauvais nombre de colonnes à la prédiction ; `RuntimeError` si `predict`, `predict_proba`, `get_depth` ou `get_n_leaves` est appelée avant `fit`.

La vérification essaie l'exemple de la docstring, compare ton arbre à celui de scikit-learn sur des demi-lunes (trois réglages : même profondeur, même nombre de feuilles, mêmes prédictions sur une grille ; ces réglages évitent les nœuds minuscules, où deux features ont souvent le même gain : scikit-learn départage ces égalités au hasard, ta librairie par le plus petit indice), affiche les règles de ton arbre de profondeur 2 sur les manchots, puis lance les tests (ceux du 13.25 restent pour plus tard).

Dans tes notes : ta construction est-elle récursive ou utilise-t-elle une pile ? Combien de fois appelles-tu `best_split` pour un arbre de $L$ feuilles ? Quelle est la profondeur maximale d'un arbre sur $n$ exemples, et quel risque cela fait-il courir à une fonction récursive en Python ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="13")   # reload your saved file
with checked("13.24"):
    doc_24 = fitted(mylearn.tree.DecisionTreeClassifier(), np.array([[1.0], [2.0], [3.0], [4.0]]), np.array([0, 0, 1, 1]))
    root_24 = getattr(doc_24, "root_", None)
    print("docstring example: root", None if root_24 is None else (root_24.feature, root_24.threshold),
          "(expected (0, 2.5)) · depth", doc_24.get_depth(), "and leaves", doc_24.get_n_leaves(), "(expected 1 and 2)")
with checked("13.24"):
    X_24, Y_24 = wb.synth.make_moons(n=300, noise=0.3, seed=1324)
    X_24 = as_float32(X_24)                                       # continuous data, no ties
    grid1_24, grid2_24 = np.meshgrid(np.linspace(-1.5, 2.5, 80), np.linspace(-1.2, 1.7, 60))
    grid_24 = as_float32(np.column_stack([grid1_24.ravel(), grid2_24.ravel()]))
    for params in [{"max_depth": 3}, {"min_samples_split": 10}, {"criterion": "entropy", "min_samples_leaf": 5}]:
        mine_24 = fitted(mylearn.tree.DecisionTreeClassifier(**params), X_24, Y_24)
        oracle_24 = DecisionTreeClassifier(random_state=0, **params).fit(X_24, Y_24)
        shape_24 = (mine_24.get_depth(), mine_24.get_n_leaves())
        if shape_24 != (oracle_24.get_depth(), oracle_24.get_n_leaves()):
            print(f"❌ Ex 13.24 : {params} : scikit-learn a une profondeur de {oracle_24.get_depth()} et "
                  f"{oracle_24.get_n_leaves()} feuilles, ton arbre {shape_24[0]} et {shape_24[1]} ; les tests "
                  "ci-dessous disent où.")
            continue
        pred_24 = mine_24.predict(grid_24)
        if not returned("13.24", "predict", pred_24):
            continue
        pred_24 = np.asarray(pred_24)
        if pred_24.shape != (len(grid_24),):
            print(f"❌ Ex 13.24 : {params} : predict doit rendre un tableau ({len(grid_24)},), un label par ligne ; "
                  f"reçu la forme {pred_24.shape}.")
        elif not np.array_equal(pred_24, oracle_24.predict(grid_24)):
            print(f"❌ Ex 13.24 : {params} : même profondeur et mêmes feuilles que scikit-learn, mais d'autres "
                  "prédictions sur la grille : un seuil ou une feature diffère ; les tests ci-dessous disent où.")
        else:
            print(f"✅ Ex 13.24 : {params} : même profondeur ({shape_24[0]}), même nombre de feuilles ({shape_24[1]}) "
                  "et mêmes prédictions que scikit-learn.")
with checked("13.24"):
    penguin_tree_24 = fitted(mylearn.tree.DecisionTreeClassifier(max_depth=2), P_TRAIN[MEASURES].to_numpy(),
                             P_TRAIN["species"].to_numpy())
    print("\n".join(tree_rules(penguin_tree_24.root_, MEASURES, penguin_tree_24.classes_)))
    print("test accuracy of this depth-2 tree:", fr(penguin_tree_24.score(P_TEST[MEASURES].to_numpy(), P_TEST["species"].to_numpy())))
if written(lambda: mylearn.tree.DecisionTreeClassifier().fit(np.array([[0.0], [1.0]]), np.array([0, 1]))):
    run_mylearn_tests("tree", keyword="test_decision_tree_classifier_ and not ensemble")
else:
    print("⏳ Ex 13.24 : les tests attendent le fit de ton DecisionTreeClassifier.")

### Ex 13.25 — Préparer l'arbre pour les ensembles : poids, features tirées, importances 🔨 ★★★ ⏱️ 50 min
**Objectif :** ajouter à ton arbre les options dont les ensembles du ch. 14 auront besoin.  
**Prérequis :** Ex 13.24 · fiche §13.5.2 et §13.5.3 · **Fil rouge :** synthétique · **mylearn :** `tree.py` · **Parcours :** C

> **mylearn** : dans `mon_travail/mylearn/tree.py`, mêmes règles qu'en 13.20. Enregistre, puis relance la cellule de vérification.

Complète `DecisionTreeClassifier` avec ce dont le ch. 14 aura besoin. **Contrat** (la docstring, plus ces précisions que vérifient les tests) :
- `sample_weight` (pour AdaBoost) : des poids positifs ou nuls, de forme `(n,)`, de somme non nulle (`ValueError` sinon) ; ils entrent dans les proportions (`value`), les impuretés et les gains, mais pas dans `n_samples` ni dans les comptes de `min_samples_split` et `min_samples_leaf` ; un poids 2 équivaut à un exemple répété deux fois, multiplier tous les poids par une constante ne change rien, et `fit` ne modifie pas `sample_weight` ;
- `max_features` (pour les forêts) : `None` (toutes les $p$ features), un entier $k$ de 1 à $p$, un réel $f$ dans $]0, 1]$ (`max(1, int(f * p))`), `"sqrt"` (`max(1, int(sqrt(p)))`) ou `"log2"` (`max(1, int(log2(p)))`), `ValueError` sinon ; à **chaque** nœud qu'on tente de couper, on tire $k$ features sans remise avec `rng.choice(p, size=k, replace=False)`, où `rng = np.random.default_rng(random_state)` est créé **une fois** dans `fit` ; on construit l'arbre en profondeur, enfant gauche d'abord ; avec `max_features=None` et `splitter="best"`, rien n'est tiré ;
- `splitter="random"` (pour les ExtraTrees) : transmis à `best_split` avec le même `rng` ; `ValueError` pour une autre valeur que `"best"` ou `"random"` ;
- `feature_importances_` : pour chaque feature, la somme, sur les nœuds coupés selon elle, de $\frac{w_{\text{nœud}}}{w_{\text{racine}}} \times \text{gain}$, puis normalisée pour sommer à 1 (des zéros si la racine est une feuille).

La vérification compare des poids entiers à des exemples répétés, tes importances à celles de scikit-learn, vérifie que `random_state` rend `max_features` reproductible, puis lance les tests.

Dans tes notes : pourquoi créer le générateur dans `fit` plutôt que dans `__init__` ? Sur ces données (deux features utiles, trois de bruit), que valent les importances des features de bruit, et pourquoi ne sont-elles pas nulles ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="13")   # reload your saved file
with checked("13.25"):
    X_25, Y_25 = wb.synth.make_moons(n=200, noise=0.3, seed=1325)
    rng_25 = np.random.default_rng(1325)
    X_25 = as_float32(np.hstack([X_25, rng_25.normal(size=(200, 3))]))   # 2 useful features, then 3 of noise
    W_25 = rng_25.integers(1, 4, size=200).astype(float)                   # weights 1, 2 or 3
    weighted_25 = fitted(mylearn.tree.DecisionTreeClassifier(max_depth=4), X_25, Y_25, sample_weight=W_25)
    repeated_25 = fitted(mylearn.tree.DecisionTreeClassifier(max_depth=4), np.repeat(X_25, W_25.astype(int), axis=0),
                         np.repeat(Y_25, W_25.astype(int)))
    verdict("13.25", np.allclose(weighted_25.predict_proba(X_25), repeated_25.predict_proba(X_25)),
            "des poids 1, 2 ou 3 donnent le même arbre que des exemples répétés 1, 2 ou 3 fois.",
            "les poids ne font pas le même arbre que des exemples répétés : relis comment ils entrent dans les "
            "proportions, les impuretés et les gains.")
with checked("13.25"):
    mine_25 = fitted(mylearn.tree.DecisionTreeClassifier(max_depth=5), X_25, Y_25)
    oracle_25 = DecisionTreeClassifier(max_depth=5, random_state=0).fit(X_25, Y_25)
    print("feature_importances_ (yours):        ", np.round(np.asarray(mine_25.feature_importances_, dtype=float), 4).tolist())
    print("feature_importances_ (scikit-learn): ", np.round(oracle_25.feature_importances_, 4).tolist())
    verdict("13.25", np.allclose(mine_25.feature_importances_, oracle_25.feature_importances_),
            "mêmes importances que scikit-learn.", "tes importances diffèrent de celles de scikit-learn.")
with checked("13.25"):
    grid_25 = as_float32(np.random.default_rng(25).normal(size=(500, 5)))
    first_25 = mylearn.tree.DecisionTreeClassifier(max_features="sqrt", random_state=0)
    first_predictions_25 = fitted(first_25, X_25, Y_25).predict(grid_25)
    refit_predictions_25 = fitted(first_25, X_25, Y_25).predict(grid_25)            # the same object, fitted again
    other_25 = fitted(mylearn.tree.DecisionTreeClassifier(max_features="sqrt", random_state=1), X_25, Y_25)
    verdict("13.25", np.array_equal(first_predictions_25, refit_predictions_25)
            and not np.array_equal(first_predictions_25, other_25.predict(grid_25)),
            "max_features=\"sqrt\" : un second fit du même arbre redonne le même arbre, un autre random_state en "
            "donne un autre.",
            "max_features=\"sqrt\" : un second fit avec le même random_state doit redonner le même arbre (le "
            "générateur est créé dans fit, pas dans __init__), et un autre random_state un autre arbre.")
if written(lambda: mylearn.tree.DecisionTreeClassifier(max_features=1, random_state=0).fit(np.array([[0.0], [1.0]]), np.array([0, 1]))):
    run_mylearn_tests("tree", keyword="test_decision_tree_classifier_ensemble_")
else:
    print("⏳ Ex 13.25 : les tests attendent le fit de ton DecisionTreeClassifier.")

### Ex 13.26 — Arbre de régression : même squelette, autre critère 🔨 ★★ ⏱️ 30 min
**Objectif :** réutiliser la construction de l'arbre pour une cible réelle, avec la variance pour impureté.  
**Prérequis :** Ex 13.24 · ch. 9 (MSE, R²) · fiche §13.5 (arbres de régression) · **Fil rouge :** California · **mylearn :** `tree.py` · **Parcours :** M, C

> **mylearn** : dans `mon_travail/mylearn/tree.py`, mêmes règles qu'en 13.20. Enregistre, puis relance la cellule de vérification.

Écris la classe `DecisionTreeRegressor`. **Contrat** (la docstring, plus ces précisions que vérifient les tests) : la même construction que `DecisionTreeClassifier` (mêmes règles d'arrêt, mêmes hyperparamètres, mêmes erreurs), avec ces différences :
- `criterion="squared_error"` seulement (`ValueError` pour toute autre valeur, `"gini"` compris) : l'impureté d'un nœud est sa variance (`variance_impurity`), et une feuille est « pure » quand ses cibles sont toutes égales ;
- la `value` d'un nœud est un tableau de forme `(1,)` : la moyenne (pondérée) des cibles qui l'atteignent ;
- `predict(X)` rend un tableau `float64` de forme `(n,)` ; il n'y a ni `classes_` ni `predict_proba` ; `score(X, y)` rend le $R^2$, un `float` Python.

La vérification essaie l'exemple de la docstring, compare ton arbre à celui de scikit-learn sur 2 000 districts de California (deux réglages), affiche les règles d'un arbre de profondeur 2, puis lance les tests.

Dans tes notes : que prédit ton arbre pour un district dont le revenu médian dépasse tous ceux de l'entraînement ? Pourquoi un arbre de régression sans limite a-t-il un $R^2$ d'entraînement de 1 (ou presque) ?

In [ ]:
CAL_26 = wb.datasets.load_california()
CAL_FEATURES_26 = [column for column in CAL_26.columns if column != "MedHouseVal"]
rows_26 = np.random.default_rng(1326).choice(len(CAL_26), 3000, replace=False)       # 3,000 districts
X_CAL_26 = as_float32(CAL_26[CAL_FEATURES_26].to_numpy()[rows_26])
Y_CAL_26 = CAL_26["MedHouseVal"].to_numpy()[rows_26]
X_TRAIN_26, X_TEST_26, Y_TRAIN_26, Y_TEST_26 = train_test_split(X_CAL_26, Y_CAL_26, test_size=1000, random_state=1326)

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="13")   # reload your saved file
with checked("13.26"):
    doc_26 = fitted(mylearn.tree.DecisionTreeRegressor(max_depth=1), np.array([[1.0], [2.0], [3.0], [4.0]]),
                    np.array([1.0, 1.2, 3.0, 3.4]))
    print("docstring example:", np.round(np.asarray(doc_26.predict(np.array([[0.0], [5.0]])), dtype=float), 4).tolist(),
          "(expected [1.1, 3.2])")
with checked("13.26"):
    for params in [{"max_depth": 5}, {"min_samples_leaf": 50}]:
        start_26 = time.perf_counter()
        mine_26 = fitted(mylearn.tree.DecisionTreeRegressor(**params), X_TRAIN_26, Y_TRAIN_26)
        seconds_26 = time.perf_counter() - start_26
        oracle_26 = DecisionTreeRegressor(random_state=0, **params).fit(X_TRAIN_26, Y_TRAIN_26)
        same_26 = (mine_26.get_depth() == oracle_26.get_depth() and mine_26.get_n_leaves() == oracle_26.get_n_leaves()
                   and np.allclose(np.ravel(mine_26.predict(X_TEST_26)), oracle_26.predict(X_TEST_26)))
        verdict("13.26", same_26,
                f"{params} : même arbre que scikit-learn ({oracle_26.get_n_leaves()} feuilles, R² de test "
                f"{fr(oracle_26.score(X_TEST_26, Y_TEST_26))}), construit en {fr(seconds_26, 2)} s.",
                f"{params} : scikit-learn a une profondeur de {oracle_26.get_depth()} et {oracle_26.get_n_leaves()} "
                f"feuilles, ton arbre {mine_26.get_depth()} et {mine_26.get_n_leaves()} (ou d'autres prédictions) ; "
                "les tests ci-dessous disent où.")
with checked("13.26"):
    small_26 = fitted(mylearn.tree.DecisionTreeRegressor(max_depth=2), X_TRAIN_26, Y_TRAIN_26)
    print("\n".join(tree_rules(small_26.root_, CAL_FEATURES_26)))
if written(lambda: mylearn.tree.DecisionTreeRegressor().fit(np.array([[0.0], [1.0]]), np.array([0.0, 1.0]))):
    run_mylearn_tests("tree", keyword="test_decision_tree_regressor_ and not ensemble")
else:
    print("⏳ Ex 13.26 : les tests attendent le fit de ton DecisionTreeRegressor.")

## Partie D · Naïve Bayes

Naïve Bayes gaussien sur deux jeux de points (13.27), puis ta librairie `mylearn/naive_bayes.py` : `GaussianNB` en log-probabilités (13.28) et `MultinomialNB` avec lissage, sur des lettres de Holmes et de Verne (13.29). Enfin, un vrai problème de texte : reconnaître les phrases de dialogue de Holmes avec un sac de mots (13.30).

### Ex 13.27 — Naïve Bayes sur des nuages gaussiens puis sur des lunes 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir la frontière d'un Naïve Bayes gaussien et ce que des données en plus lui apportent.  
**Prérequis :** Ex 13.12 · fiche §13.6 (Naïve Bayes gaussien) · **Fil rouge :** synthétique · **Parcours :** C

Deux expériences avec `GaussianNB()` de scikit-learn. Les **nuages gaussiens** : 300 points de la classe 0 autour de (0, 0), d'écart-type 0,5 ; 300 points de la classe 1 autour de (1,5 ; 0), d'écart-type 1,5 (trois fois plus étalés). Les **lunes** : deux demi-lunes peu bruitées (bruit 0,2), dont on garde $n$ points pour l'entraînement et 2 000 pour le test, 10 tirages par valeur de $n$.

Prédis, **avant** d'exécuter quoi que ce soit :
a) `surround_27` : la frontière sur les nuages gaussiens : `"A"` une droite entre les deux nuages ; `"B"` une courbe fermée autour de la classe 0 ; `"C"` une courbe fermée autour de la classe 1 ; `"D"` deux droites parallèles, la classe 0 entre elles ;
b) `moons_27` : l'accuracy de test moyenne sur les lunes avec 200 points d'entraînement : `"A"` environ 0,5 ; `"B"` environ 0,7 ; `"C"` environ 0,86 ; `"D"` environ 0,97 ;
c) `more_data_27` : passer de 200 à 5 000 points d'entraînement fait-il gagner plus de 3 points d'accuracy sur les lunes ? (`True` ou `False`)

Puis exécute l'expérience.

In [ ]:
X_BLOBS_27, Y_BLOBS_27 = wb.synth.make_blobs(n=600, centers=[[0.0, 0.0], [1.5, 0.0]], std=[0.5, 1.5], seed=1327)

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
surround_27 = ...    # a) "A", "B", "C" or "D"
moons_27 = ...       # b) "A", "B", "C" or "D"
more_data_27 = ...   # c) True or False

In [ ]:
if any(answer is ... or answer is None for answer in [surround_27, moons_27, more_data_27]):
    print("⏳ Ex 13.27 : écris d'abord tes trois prédictions, puis relance cette cellule.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    blobs_27 = GaussianNB().fit(X_BLOBS_27, Y_BLOBS_27)
    wb.plot.plot_decision_boundary(blobs_27, X_BLOBS_27, Y_BLOBS_27, ax=axes[0], title="GaussianNB, two blobs")
    for n in [50, 200, 5000]:
        accuracies = []
        for r in range(10):                                   # 10 draws: n training points, 2,000 test points
            X, y = wb.synth.make_moons(n=n + 2000, noise=0.2, seed=1327 + r)
            accuracies.append(GaussianNB().fit(X[:n], y[:n]).score(X[n:], y[n:]))
        print(f"moons, {n:4d} training points: mean test accuracy {np.mean(accuracies):.3f} "
              f"(from {min(accuracies):.3f} to {max(accuracies):.3f})")
    X_moons_27, Y_moons_27 = wb.synth.make_moons(n=2200, noise=0.2, seed=1327)
    wb.plot.plot_decision_boundary(GaussianNB().fit(X_moons_27[:200], Y_moons_27[:200]), X_moons_27[200:700],
                                   Y_moons_27[200:700], ax=axes[1], title="GaussianNB, moons (200 training points)")
    plt.show()
    wb.check("13.27a", surround_27)
    wb.check("13.27b", moons_27)
    wb.check("13.27c", more_data_27)

Dans tes notes : compare avec tes prédictions. Sur les nuages gaussiens, pourquoi la frontière a-t-elle cette forme ? Sur les lunes, quelles erreurs restent, et que font les données en plus ? Les erreurs viennent-elles du biais ou de la variance ?

### Ex 13.28 — Coder GaussianNB avec log-densités et logsumexp 🔨 ★★★ ⏱️ 40 min
**Objectif :** écrire Naïve Bayes gaussien entièrement en log-probabilités, sans jamais multiplier des probabilités.  
**Prérequis :** ✏️ 13.7 · ch. 4 (règle de Bayes, logsumexp en 4.18) · ch. 2 (loi normale) · fiche §13.6 · **Fil rouge :** Penguins · **mylearn :** `naive_bayes.py` · **Parcours :** M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/naive_bayes.py` (créé par `python tools/start_chapter.py 13`), lis la docstring de chaque fonction et de chaque méthode, remplace les `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy seulement : scikit-learn et SciPy sont les **oracles** des tests. La cellule de vérification recharge ta librairie, compare quelques résultats à scikit-learn, puis lance les tests de l'exercice ; `python -m pytest tests/test_ch13_naive_bayes.py -q` les lance tous dans un terminal.

Écris `log_gaussian_pdf`, `logsumexp` et la classe `GaussianNB`. **Contrat** (les docstrings, plus ces précisions que vérifient les tests) :
- `log_gaussian_pdf(x, mean, var)` rend, élément par élément et avec le broadcasting, $-\frac12 \log(2\pi\sigma^2) - \frac{(x - \mu)^2}{2\sigma^2}$ ; le résultat reste fini même très loin de la moyenne (les tests l'éprouvent) ; `ValueError` si une variance est négative ou nulle ; la fonction ne modifie pas ses arguments ;
- `logsumexp(a, axis=-1)` rend $\log \sum e^{a}$ le long de l'axe (qui disparaît), sans dépassement ni sous-dépassement, même pour des valeurs autour de ±1 000 ; un `-np.inf` parmi des valeurs finies compte pour 0 dans la somme ; ne modifie pas `a` ;
- `GaussianNB(var_smoothing=1e-9, priors=None)` : `__init__` range ses deux arguments ; `fit(X, y)` apprend `classes_` (triés), `class_count_`, `class_prior_` (les fréquences des classes, ou `priors` s'il est donné), `theta_` (les moyennes par classe) et `var_` (les variances par classe, ddof = 0, **plus** `epsilon_`), avec `epsilon_` = `var_smoothing` × la plus grande variance des features calculée sur tout `X`, et renvoie `self` ; un second `fit` oublie le premier ; `fit` ne modifie pas ses arguments ;
- `predict_joint_log_proba(X)` rend la matrice `(n, K)` des $\log P(c) + \sum_j \log \mathcal{N}(x_j ; \theta_{cj}, \sigma^2_{cj})$ ; `predict_log_proba` la normalise avec **ton** `logsumexp` ; `predict_proba` en prend l'exponentielle ; `predict` rend la classe de plus grand posterior ; `score` l'accuracy, un `float` Python ;
- erreurs : `ValueError` si `X` n'est pas 2-D ou contient `NaN` (comme scikit-learn), si `len(X) != len(y)`, ou si `priors` n'a pas une valeur positive ou nulle par classe, de somme 1 à $10^{-6}$ près ; `RuntimeError` si `predict_joint_log_proba` est appelée avant `fit` (les autres méthodes passent par elle).

La vérification essaie les exemples des docstrings, compare tes moyennes, tes variances et tes posteriors à ceux de scikit-learn sur les manchots, montre ce que donnerait le produit des probabilités sur les 784 pixels de MNIST, puis lance les tests.

Dans tes notes : sur MNIST, pourquoi le produit des probabilités donne-t-il `nan` ? À quoi sert `epsilon_` (pense à un pixel toujours noir dans une classe) ? Pourquoi Naïve Bayes gaussien classe-t-il si mal les chiffres, comparé à un kNN ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="13")   # reload your saved file
with checked("13.28"):
    doc_28 = mylearn.naive_bayes.log_gaussian_pdf(np.array([0.0, 1.0]), 0.0, 1.0)
    if returned("13.28", "log_gaussian_pdf", doc_28):
        print("docstring example: log_gaussian_pdf([0, 1], 0, 1) =", np.round(np.asarray(doc_28, dtype=float), 4).tolist(),
              "(expected [-0.9189, -1.4189])")
with checked("13.28"):
    lse_28 = mylearn.naive_bayes.logsumexp(np.array([-1000.0, -1000.0]))
    if returned("13.28", "logsumexp", lse_28):
        print("docstring example: logsumexp([-1000, -1000]) =", round(float(lse_28), 4), "(expected -999.3069)")
with checked("13.28"):
    X_train_28, y_train_28 = P_TRAIN[MEASURES].to_numpy(), P_TRAIN["species"].to_numpy()
    X_test_28 = P_TEST[MEASURES].to_numpy()
    mine_28 = fitted(mylearn.naive_bayes.GaussianNB(), X_train_28, y_train_28)
    oracle_28 = GaussianNB().fit(X_train_28, y_train_28)
    proba_28 = mine_28.predict_proba(X_test_28)
    if returned("13.28", "predict_proba", proba_28):
        verdict("13.28", np.allclose(mine_28.theta_, oracle_28.theta_) and np.allclose(mine_28.var_, oracle_28.var_)
                and np.allclose(proba_28, oracle_28.predict_proba(X_test_28)),
                f"manchots : mêmes moyennes, mêmes variances et mêmes posteriors que scikit-learn (accuracy de test "
                f"{fr(oracle_28.score(X_test_28, P_TEST['species']))}).",
                "manchots : tes moyennes, tes variances ou tes posteriors diffèrent de ceux de scikit-learn ; les tests "
                "ci-dessous disent où.")
with checked("13.28"):
    X_mnist_28, y_mnist_28 = wb.datasets.load_mnist("train", n=3000, flatten=True, seed=1328)
    X_mnist_28 = X_mnist_28 / 255.0
    digits_28 = fitted(mylearn.naive_bayes.GaussianNB(), X_mnist_28[:2500], y_mnist_28[:2500])
    joint_28 = np.asarray(digits_28.predict_joint_log_proba(X_mnist_28[2500:]), dtype=float)
    with np.errstate(divide="ignore", invalid="ignore", over="ignore", under="ignore"):
        products_28 = np.exp(joint_28)                                    # what multiplying probabilities gives
        naive_28 = products_28 / products_28.sum(axis=1, keepdims=True)
    print(f"MNIST, 784 pixels: joint log-probabilities from {joint_28.min():.3g} to {joint_28.max():.3g}")
    print(f"multiply, then normalise: {int(np.isnan(naive_28).any(axis=1).sum())} of the 500 test images get nan")
    digits_proba_28 = digits_28.predict_proba(X_mnist_28[2500:])
    if returned("13.28", "predict_proba", digits_proba_28):
        digits_proba_28 = np.asarray(digits_proba_28, dtype=float)
        verdict("13.28", np.isfinite(digits_proba_28).all() and np.allclose(digits_proba_28.sum(axis=1), 1),
                f"en log-probabilités avec logsumexp : aucune image sans posterior (accuracy "
                f"{fr(digits_28.score(X_mnist_28[2500:], y_mnist_28[2500:]))}).",
                "tes posteriors contiennent des nan ou ne somment pas à 1 : normalise les log-probabilités jointes "
                "avec logsumexp avant de prendre l'exponentielle.")
if written(lambda: mylearn.naive_bayes.log_gaussian_pdf(0.0, 0.0, 1.0)):
    run_mylearn_tests("naive_bayes", keyword="test_log_gaussian_pdf_ or test_logsumexp_ or test_gaussian_nb_")
else:
    print("⏳ Ex 13.28 : les tests attendent ton log_gaussian_pdf.")

### Ex 13.29 — Coder MultinomialNB avec lissage 🔨 ★★ ⏱️ 30 min
**Objectif :** écrire Naïve Bayes multinomial pour des comptes, avec le lissage de Laplace.  
**Prérequis :** ✏️ 13.6 · Ex 13.28 · fiche §13.6 (lissage) · **Fil rouge :** Holmes/Verne · **mylearn :** `naive_bayes.py` · **Parcours :** C

> **mylearn** : dans `mon_travail/mylearn/naive_bayes.py`, mêmes règles qu'en 13.28. Enregistre, puis relance la cellule de vérification.

Écris la classe `MultinomialNB`. **Contrat** (la docstring, plus ces précisions que vérifient les tests) :
- `MultinomialNB(alpha=1.0, fit_prior=True)` : `__init__` range ses deux arguments ; `fit(X, y)` reçoit des **comptes** (positifs ou nuls) et apprend `classes_` (triés), `class_count_`, `feature_count_` (la matrice `(K, d)` des comptes totaux $N_{cj}$ de chaque feature dans chaque classe), `class_log_prior_` (le log des fréquences des classes, ou $-\log K$ partout si `fit_prior=False`) et `feature_log_prob_` $= \log \frac{N_{cj} + \alpha}{N_c + \alpha d}$, puis renvoie `self` ; un second `fit` oublie le premier ; `fit` ne modifie pas ses arguments ;
- `alpha=0` est permis : les fréquences brutes (un compte nul donne $\log 0 = -\infty$) ;
- `predict_joint_log_proba(X)` $=$ `X @ feature_log_prob_.T + class_log_prior_` ; `predict_log_proba`, `predict_proba`, `predict` et `score` comme pour `GaussianNB` (avec ton `logsumexp`) : des documents très longs ne donnent ni `nan` ni 0 partout ;
- erreurs : `ValueError` si `X` a une valeur négative ou `NaN`, n'est pas 2-D, si `len(X) != len(y)`, ou si `alpha < 0` ; `RuntimeError` si `predict_joint_log_proba` est appelée avant `fit`.

Les données : Holmes (anglais, `"en"`) et Verne (français, `"fr"`) découpés en morceaux de quelques caractères ; chaque morceau devient les 26 comptes de ses lettres a à z (`letter_counts`). La vérification essaie l'exemple de la docstring, compare tes probabilités à celles de scikit-learn sur des morceaux de 20 caractères, mesure ton accuracy pour quatre longueurs de morceaux, puis lance les tests.

Dans tes notes : comment l'accuracy varie-t-elle avec la longueur des morceaux, et pourquoi ? Sans lissage, qu'arriverait-il à un morceau anglais qui contient une lettre absente de tous les morceaux français d'entraînement (si le « w », rare en français, n'y apparaissait jamais, par exemple) ?

In [ ]:
def letter_counts(text, size):
    """(n_chunks, 26): the counts of the letters a to z in the consecutive chunks of `size` characters of the text
    (lower-cased; the other characters, accented letters included, are ignored)."""
    text = text.lower()
    n_chunks = len(text) // size
    codes = np.frombuffer(text[:n_chunks * size].encode("utf-32-le"), dtype=np.uint32).reshape(n_chunks, size)
    return np.stack([(codes == ord("a") + j).sum(axis=1) for j in range(26)], axis=1)


HOLMES_29, VERNE_29 = wb.datasets.load_holmes(), wb.datasets.load_verne()


def chunks_29(size):
    """Training and test parts (70/30) of the chunks of `size` characters of Holmes ("en") and Verne ("fr")."""
    english, french = letter_counts(HOLMES_29, size), letter_counts(VERNE_29, size)
    X = np.vstack([english, french])
    y = np.array(["en"] * len(english) + ["fr"] * len(french))
    return train_test_split(X, y, test_size=0.3, random_state=1329, stratify=y)

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="13")   # reload your saved file
with checked("13.29"):
    X_doc_29 = np.array([[2, 1, 0], [3, 0, 1], [0, 2, 3], [1, 0, 4]])
    doc_29 = fitted(mylearn.naive_bayes.MultinomialNB(alpha=1.0), X_doc_29, np.array(["en", "en", "fr", "fr"]))
    print("docstring example: P(feature | class) =", np.round(np.exp(doc_29.feature_log_prob_), 4).tolist(),
          "(expected [[0.6, 0.2, 0.2], [0.1538, 0.2308, 0.6154]])")
with checked("13.29"):
    X_train_29, X_test_29, y_train_29, y_test_29 = chunks_29(20)
    mine_29 = fitted(mylearn.naive_bayes.MultinomialNB(), X_train_29, y_train_29)
    oracle_29 = MultinomialNB(alpha=1.0, force_alpha=True).fit(X_train_29, y_train_29)
    verdict("13.29", np.allclose(mine_29.feature_log_prob_, oracle_29.feature_log_prob_)
            and np.allclose(mine_29.predict_proba(X_test_29), oracle_29.predict_proba(X_test_29)),
            f"morceaux de 20 caractères : mêmes probabilités que scikit-learn ({len(X_train_29)} morceaux "
            "d'entraînement).",
            "morceaux de 20 caractères : tes probabilités diffèrent de celles de scikit-learn ; les tests ci-dessous "
            "disent où.")
    for size in [10, 20, 50, 200]:
        X_tr, X_te, y_tr, y_te = chunks_29(size)
        model = fitted(mylearn.naive_bayes.MultinomialNB(), X_tr, y_tr)
        print(f"chunks of {size:3d} characters: test accuracy {model.score(X_te, y_te):.4f} ({len(X_te)} test chunks)")
if written(lambda: mylearn.naive_bayes.MultinomialNB().fit(np.array([[1, 0], [0, 1]]), np.array([0, 1]))):
    run_mylearn_tests("naive_bayes", keyword="test_multinomial_nb_")
else:
    print("⏳ Ex 13.29 : les tests attendent le fit de ton MultinomialNB.")

### Ex 13.30 — Dialogue ou narration ? Classer des phrases de Holmes : sac de mots, TF-IDF et Naïve Bayes 📦 ★★★ ⏱️ 40 min
**Objectif :** classer des textes avec un sac de mots, et lire ce que le modèle a appris.  
**Prérequis :** Ex 13.12 · ch. 6 (fréquences) · ch. 3 (accuracy, classe majoritaire) · fiche §13.6 (Naïve Bayes sur des textes) · **Fil rouge :** Holmes/Verne · **Parcours :** R, C

La cellule suivante découpe les nouvelles de Holmes en phrases et donne à chacune un label : 1 (**dialogue**) si toutes ses lettres sont entre guillemets « “ » et « ” », 0 (**narration**) si aucune ne l'est ; les phrases qui mêlent les deux sont écartées, et les guillemets retirés du texte : le modèle doit reconnaître le dialogue à ses **mots**. 70 % des phrases servent à l'entraînement. Avec `make_pipeline`, calcule :
a) `dialogue_share_30` : la part des phrases de dialogue parmi toutes les phrases ;
b) `baseline_30` : l'accuracy de test d'un modèle qui prédit toujours la classe majoritaire de l'entraînement ;
c) `count_nb_30` : l'accuracy de test de `make_pipeline(CountVectorizer(), MultinomialNB())` entraîné sur `S_TRAIN_30` ;
d) `tfidf_nb_30` : la même chose avec `TfidfVectorizer()` à la place de `CountVectorizer()` ;
e) `tfidf_lr_30` : la même chose avec `TfidfVectorizer()` et `LogisticRegression(max_iter=1000)` ;
f) `top_dialogue_30` : les 5 mots qui ont le plus grand écart `feature_log_prob_[1] - feature_log_prob_[0]` dans le pipeline de c) (une liste de chaînes, dans n'importe quel ordre ; `get_feature_names_out()` donne les mots).

Dans tes notes : pourquoi le dialogue est-il si majoritaire dans Holmes (pense à qui raconte les affaires) ? Que vaut c) comparé à b) ? Pourquoi TF-IDF fait-il **moins bien** avec Naïve Bayes qu'avec la régression logistique (regarde ce que deviennent les « comptes » d'une phrase, et le poids du prior) ? Les mots de f) te surprennent-ils ?

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import make_pipeline


def holmes_sentences():
    """The sentences of Holmes, each labelled 1 (dialogue: all its letters between “ and ”) or 0 (narration: none of
    them); the sentences that mix both, or have fewer than 4 words, are left out, and the quotes are removed."""
    text = wb.datasets.load_holmes()
    text = text[text.index("To Sherlock Holmes she is always"):]           # skip the title and the contents
    boundary = re.compile(r"(?:(?<=[.!?])|(?<=[.!?]”))\s+")                # a space after . ! or ?, possibly after ”
    sentences, labels = [], []
    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = paragraph.replace("\n", " ").strip()
        if len(paragraph.split()) < 4 or paragraph.isupper():              # headings
            continue
        inside, state = np.zeros(len(paragraph), dtype=bool), False        # a paragraph starts outside the quotes
        for i, char in enumerate(paragraph):
            if char == "“":
                state = True
            inside[i] = state
            if char == "”":
                state = False
        start = 0
        for piece in boundary.split(paragraph):
            start = paragraph.index(piece, start)
            letters = [inside[i] for i in range(start, start + len(piece)) if paragraph[i].isalpha()]
            start += len(piece)
            clean = piece.replace("“", "").replace("”", "").strip()
            if len(clean.split()) >= 4 and letters and (all(letters) or not any(letters)):
                sentences.append(clean)
                labels.append(int(all(letters)))
    return sentences, np.array(labels)


SENTENCES_30, LABELS_30 = holmes_sentences()
S_TRAIN_30, S_TEST_30, Y_TRAIN_30, Y_TEST_30 = train_test_split(SENTENCES_30, LABELS_30, test_size=0.3,
                                                                random_state=1330, stratify=LABELS_30)
print(len(SENTENCES_30), "sentences, for example:")
for label in (1, 0):
    for k in np.flatnonzero(LABELS_30 == label)[:2]:
        print(f"  {label}  {SENTENCES_30[k][:100]}")

In [ ]:
dialogue_share_30 = ...  # a) the share of dialogue sentences (label 1) among all of SENTENCES_30
baseline_30 = ...        # b) the test accuracy of a model that always predicts the majority class of the training part
count_nb_30 = ...        # c) the test accuracy of make_pipeline(CountVectorizer(), MultinomialNB()) fitted on S_TRAIN_30
tfidf_nb_30 = ...        # d) the same with TfidfVectorizer() instead of CountVectorizer()
tfidf_lr_30 = ...        # e) the same with TfidfVectorizer() and LogisticRegression(max_iter=1000)
top_dialogue_30 = ...    # f) the 5 words with the largest feature_log_prob_[1] - feature_log_prob_[0] in the pipeline
                         #    of c) (a list of strings, in any order)

In [ ]:
wb.check("13.30a", dialogue_share_30)
wb.check("13.30b", baseline_30)
wb.check("13.30c", count_nb_30)
wb.check("13.30d", tfidf_nb_30)
wb.check("13.30e", tfidf_lr_30)
wb.check("13.30f", top_dialogue_30)

## Partie E · La régression logistique

Le classifieur paramétrique de référence, absent du livre à ce chapitre : celui de scikit-learn, ses coefficients et ses probabilités (13.31), puis le tien, dans `mylearn/logistic.py` (13.32). Les deux exercices séparent les Adélie des Chinstrap ; la cellule suivante garde ces 214 manchots et les découpe (70/30).

In [ ]:
# Adélie or Chinstrap? The 214 penguins of these two species (70 % for training, 30 % for testing)
AC_PENGUINS = PENGUINS[PENGUINS["species"].isin(["Adelie", "Chinstrap"])]
AC_TRAIN, AC_TEST = train_test_split(AC_PENGUINS, test_size=0.3, random_state=1331, stratify=AC_PENGUINS["species"])
print(len(AC_TRAIN), "training penguins,", len(AC_TEST), "test penguins")

### Ex 13.31 — Régression logistique scikit-learn : coefficients et probabilités 📦 ★★ ⏱️ 25 min
**Objectif :** lire les coefficients d'une régression logistique en odds, et voir l'effet de sa pénalité par défaut.  
**Prérequis :** Ex 13.12 · ch. 12 (standardisation) · fiche « Au-delà du livre : la régression logistique » · **Fil rouge :** Penguins · **Parcours :** R, C

Les 4 mesures des manchots Adélie et Chinstrap, standardisées sur l'entraînement (`Z_TRAIN_31`). Avec `LogisticRegression` de scikit-learn, calcule :
- `model_31` : `LogisticRegression()`, avec ses réglages par défaut, entraîné sur `Z_TRAIN_31`, `Y_TRAIN_31` ;
- a) `coef_31` : ses 4 coefficients, dans l'ordre de `MEASURES` ;
- b) `odds_31` : le facteur qui multiplie les **odds** $P(\text{Chinstrap}) / P(\text{Adélie})$ quand la longueur du bec augmente d'un écart-type, les autres mesures restant fixes ;
- c) `proba_31` : la probabilité que le nouveau manchot `NEW_31` (mesures brutes : bec de 45 mm de long et 17,5 mm d'épaisseur, nageoires de 195 mm, 3 700 g) soit un Chinstrap ;
- d) `norms_31` : la liste `[‖coef‖ avec C = 1, ‖coef‖ avec C = 100]` de `LogisticRegression(C=C, max_iter=10_000)`.

Dans tes notes : quelle mesure compte le plus, et le signe de chaque coefficient te paraît-il logique (compare les moyennes des deux espèces) ? Que fait C, et que se passe-t-il avec `C=np.inf` sur ces données (essaie : regarde la norme et l'accuracy d'entraînement) ? Pourquoi standardiser avant de comparer des coefficients ?

In [ ]:
SCALER_31 = StandardScaler().fit(AC_TRAIN[MEASURES])
Z_TRAIN_31, Z_TEST_31 = SCALER_31.transform(AC_TRAIN[MEASURES]), SCALER_31.transform(AC_TEST[MEASURES])
Y_TRAIN_31, Y_TEST_31 = AC_TRAIN["species"].to_numpy(), AC_TEST["species"].to_numpy()
NEW_31 = pd.DataFrame([[45.0, 17.5, 195.0, 3700.0]], columns=MEASURES)   # a new penguin, raw measures

In [ ]:
model_31 = ...   # LogisticRegression() (default settings) fitted on Z_TRAIN_31, Y_TRAIN_31
coef_31 = ...    # a) its 4 coefficients, in the order of MEASURES (a list or a 1-D array)
odds_31 = ...    # b) the factor that multiplies the odds P(Chinstrap) / P(Adelie) when the bill length grows by one
                 #    standard deviation (the other measures unchanged)
proba_31 = ...   # c) the probability that NEW_31 is a Chinstrap
norms_31 = ...   # d) [||coef|| with C = 1, ||coef|| with C = 100], for LogisticRegression(C=C, max_iter=10_000)

In [ ]:
wb.check("13.31a", coef_31)
wb.check("13.31b", odds_31)
wb.check("13.31c", proba_31)
wb.check("13.31d", norms_31)

### Ex 13.32 — Coder la régression logistique par descente de gradient 🔨 ★★★ ⏱️ 70 min
**Objectif :** écrire la régression logistique et sa descente de gradient, et retrouver exactement scikit-learn.  
**Prérequis :** ∂ 13.8 · ch. 5 (descente de gradient) · ch. 6 (log-loss, `info.log_loss`) · ch. 12 (standardisation) · fiche « Au-delà du livre : la régression logistique » · **Fil rouge :** Penguins · **mylearn :** `logistic.py` · **Parcours :** M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/logistic.py` (créé par `python tools/start_chapter.py 13`), lis la docstring de chaque fonction et de chaque méthode, remplace les `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy seulement : scikit-learn et SciPy sont les **oracles** des tests. La cellule de vérification recharge ta librairie, compare quelques résultats à scikit-learn, puis lance les tests de l'exercice ; `python -m pytest tests/test_ch13_logistic.py -q` les lance tous dans un terminal.

Écris la classe `LogisticRegression` (`sigmoid` est fournie). **Contrat** (la docstring, plus ces précisions que vérifient les tests) :
- `__init__(alpha=0.0, learning_rate=0.1, max_iter=1000, tol=1e-6, fit_intercept=True)` range ses arguments tels quels ;
- `fit(X, y)` code les labels par `classes_` (les deux labels distincts triés : `classes_[0]` → 0, `classes_[1]` → 1), part de $\mathbf{w} = \mathbf{0}$ et $b = 0$, et minimise $J(\mathbf{w}, b) =$ `log_loss(y, p)` $+ \frac{\alpha}{2}\lVert\mathbf{w}\rVert^2$, où `log_loss` est **ta** fonction du ch. 6, déjà importée par le fichier (`from .info import log_loss`) et $b$ n'est pas pénalisé ;
- `loss_history_` commence par $J$ au départ ($\ln 2$) ; chaque itération calcule les deux gradients au point courant ($\frac1n \mathbf{X}^\top(\mathbf{p} - \mathbf{y}) + \alpha\mathbf{w}$ et la moyenne de $\mathbf{p} - \mathbf{y}$), fait un pas sur $\mathbf{w}$ et $b$ ensemble, puis ajoute le nouveau $J$ ; l'entraînement s'arrête **après** le pas où $J$ a bougé de moins de `tol` (en valeur absolue), ou après `max_iter` pas ; `n_iter_` est le nombre de pas faits, et `len(loss_history_) == n_iter_ + 1` ; avec `fit_intercept=False`, $b$ reste à 0 ;
- après `fit` : `coef_` (forme `(d,)`, pas `(1, d)` comme scikit-learn), `intercept_` (un `float`), `n_iter_` (un `int`), `loss_history_` (une liste de `float`) ; `fit` renvoie `self`, oublie un `fit` précédent et ne modifie pas ses arguments ;
- `decision_function(X)` rend $\mathbf{X}\mathbf{w} + b$ ; `predict_proba(X)` la matrice `(n, 2)` $[1 - p, p]$ ; `predict(X)` rend `classes_[1]` si le score est **strictement** positif, `classes_[0]` sinon ; `score` l'accuracy, un `float` Python ;
- erreurs : `ValueError` si `y` n'a pas exactement deux classes, si `learning_rate <= 0`, `alpha < 0` ou `max_iter < 1`, si `X` n'est pas 2-D, contient `NaN` (comme scikit-learn) ou si `len(X) != len(y)` ; `RuntimeError` avant `fit`.

La vérification essaie l'exemple de la docstring, compare tes poids à ceux de scikit-learn sur deux mesures des manchots, sans pénalité (`C=np.inf`) puis avec $\alpha = 0{,}01$ ($C = 1/(\alpha n)$), trace l'objectif au fil des pas pour trois learning rates, puis lance les tests.

Dans tes notes : pourquoi la pénalité de scikit-learn s'écrit-elle $C = 1/(\alpha n)$ ? Pourquoi les poids de `LogisticRegression()` par défaut sont-ils plus petits que les tiens sans pénalité ? Que montrent les trois courbes ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="13")   # reload your saved file
if written(lambda: mylearn.logistic.LogisticRegression) and not written(
        lambda: mylearn.info.log_loss(np.array([0, 1]), np.array([0.2, 0.7]))):
    print("⏳ Ex 13.32 : ta fonction log_loss du ch. 6 (mylearn/info.py) n'est pas écrite ; ta LogisticRegression "
          "s'en sert pour son objectif : complète-la d'abord (exercices du ch. 6).")
with checked("13.32"):
    doc_32 = fitted(mylearn.logistic.LogisticRegression(), np.array([[-2.0], [-1.0], [0.0], [1.0], [2.0]]),
                    np.array([0, 1, 0, 1, 1]))
    print("docstring example: loss_history_[0] =", round(float(doc_32.loss_history_[0]), 4), "(expected 0.6931) · "
          "predict([[-3], [3]]) =", np.asarray(doc_32.predict(np.array([[-3.0], [3.0]]))).tolist(), "(expected [0, 1])")
with checked("13.32"):
    pair_32 = ["bill_depth_mm", "flipper_length_mm"]       # two measures where the two species overlap
    scaler_32 = StandardScaler().fit(AC_TRAIN[pair_32])
    Z_32, y_32 = scaler_32.transform(AC_TRAIN[pair_32]), AC_TRAIN["species"].to_numpy()
    for alpha, C, label in [(0.0, np.inf, "C=np.inf"), (0.01, 1 / (0.01 * len(Z_32)), "C=1/(alpha*n)")]:
        mine_32 = fitted(mylearn.logistic.LogisticRegression(alpha=alpha, learning_rate=0.5, max_iter=5000, tol=1e-9),
                         Z_32, y_32)
        oracle_32 = LogisticRegression(C=C, tol=1e-10, max_iter=10_000).fit(Z_32, y_32)
        coef_32 = np.asarray(mine_32.coef_, dtype=float)
        print(f"alpha = {alpha}: {mine_32.n_iter_} steps · coef_ {np.round(coef_32, 4).tolist()} (shape {coef_32.shape}), "
              f"intercept_ {float(mine_32.intercept_):.4f} · scikit-learn ({label}): "
              f"{np.round(oracle_32.coef_[0], 4).tolist()}, {oracle_32.intercept_[0]:.4f}")
        verdict("13.32", coef_32.shape == (2,) and np.allclose(coef_32, oracle_32.coef_[0], atol=1e-3)
                and np.isclose(float(mine_32.intercept_), oracle_32.intercept_[0], atol=1e-3),
                f"alpha = {alpha} : mêmes poids et même ordonnée à l'origine que scikit-learn avec {label}.",
                f"alpha = {alpha} : tes poids ou ton ordonnée à l'origine diffèrent de ceux de scikit-learn avec {label} ; "
                "les tests ci-dessous disent où.")
    default_32 = LogisticRegression().fit(Z_32, y_32)
    print("scikit-learn with its default settings (C=1.0):", np.round(default_32.coef_[0], 4).tolist())
with checked("13.32"):
    histories_32 = {rate: list(fitted(mylearn.logistic.LogisticRegression(learning_rate=rate, max_iter=300, tol=0.0),
                                      Z_32, y_32).loss_history_) for rate in [0.05, 0.5, 5.0]}
    for rate, history_32 in histories_32.items():
        plt.plot(history_32, label=f"learning_rate = {rate}")
    plt.xlabel("gradient step")
    plt.ylabel("objective J(w, b)")
    plt.legend()
    plt.show()
if written(lambda: mylearn.logistic.LogisticRegression().fit(np.array([[0.0], [1.0]]), np.array([0, 1]))):
    run_mylearn_tests("logistic", keyword="test_logistic_regression_")
else:
    print("⏳ Ex 13.32 : les tests attendent le fit de ta LogisticRegression.")

## Partie F · Synthèse : lire, mesurer, comparer

Reconnaître un classifieur à sa frontière (13.33), mesurer et accélérer le calcul des distances d'un kNN (13.34), comparer ta librairie à scikit-learn en accuracy et en temps (13.35), puis le défi : 96 % sur MNIST avec un classifieur classique (13.36).

In [ ]:
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline

### Ex 13.33 — Associer quatre frontières à leurs classifieurs 📈 ★ ⏱️ 15 min
**Objectif :** reconnaître un classifieur à la forme de sa frontière.  
**Prérequis :** Ex 13.12 · fiche §13.7 (tableau « frontière ») · **Fil rouge :** synthétique · **Parcours :** R

La cellule suivante entraîne quatre classifieurs sur les mêmes 300 points de deux demi-lunes bruitées, et trace leurs frontières dans un ordre mélangé (panneaux 1 à 4). Les classifieurs : A `KNeighborsClassifier(n_neighbors=1)` ; B `DecisionTreeClassifier(max_depth=6)` ; C `GaussianNB()` ; D `LogisticRegression()`. Réponds sans réentraîner les modèles :
a) `panels_33` : les lettres des classifieurs des panneaux 1, 2, 3 et 4, dans cet ordre (par exemple `"ABCD"`) ;
b) `perfect_33` : le numéro du panneau dont le classifieur classe correctement **tous** ses points d'entraînement.

Dans tes notes : pour chaque panneau, le détail qui t'a décidé. Lequel de ces quatre modèles choisirais-tu pour ces données, et comment le vérifierais-tu ?

In [ ]:
X_33, Y_33 = wb.synth.make_moons(n=300, noise=0.3, seed=1333)
CLASSIFIERS_33 = {"A": KNeighborsClassifier(n_neighbors=1), "B": DecisionTreeClassifier(max_depth=6, random_state=0),
                  "C": GaussianNB(), "D": LogisticRegression()}
ORDER_33 = "".join(np.random.default_rng(1302).permutation(list(CLASSIFIERS_33)))   # the order of the panels
fig, axes = plt.subplots(1, 4, figsize=(20, 4.2))
for panel, (ax, letter) in enumerate(zip(axes, ORDER_33), start=1):
    model = CLASSIFIERS_33[letter].fit(X_33, Y_33)
    wb.plot.plot_decision_boundary(model, X_33, Y_33, ax=ax, title=f"panel {panel}")
plt.show()
del ORDER_33                         # the answer is not kept in the notebook

In [ ]:
panels_33 = ...    # a) the letters of the classifiers of panels 1, 2, 3 and 4, in this order (e.g. "ABCD")
perfect_33 = ...   # b) the number (1 to 4) of the panel whose classifier classifies ALL its training points right

In [ ]:
wb.check("13.33a", panels_33)
wb.check("13.33b", perfect_33)

### Ex 13.34 — Mesurer et vectoriser : accélérer son kNN 🛠️ ★★ ⏱️ 25 min
**Objectif :** mesurer un temps d'exécution proprement, et remplacer une boucle par du calcul vectorisé.  
**Prérequis :** Ex 13.14 · 0B (produit matriciel) · fiche §13.3 (le coût d'un kNN) · **Fil rouge :** MNIST · **Parcours :** C

`TRAIN_34` : 2 000 images de MNIST, `TEST_34` : 500, pixels dans $[0, 1]$. Un kNN passe son temps à calculer des distances : trois façons de calculer la matrice `(len(A), len(B))` des distances euclidiennes, et un outil de mesure. Écris :
- `distances_loop_34(A, B)` : deux boucles Python sur les lignes (NumPy pour la distance d'une paire) ;
- `distances_broadcast_34(A, B)` : une seule différence par broadcasting, `A[:, None, :] - B[None, :, :]`, sans boucle ;
- `distances_dot_34(A, B)` : l'identité $\lVert\mathbf{a} - \mathbf{b}\rVert^2 = \lVert\mathbf{a}\rVert^2 - 2\,\mathbf{a}\cdot\mathbf{b} + \lVert\mathbf{b}\rVert^2$, avec un seul produit matriciel pour tous les $\mathbf{a}\cdot\mathbf{b}$ ; elle ne doit **jamais** rendre `nan` ;
- `timed_34(function, *args, repeat=3)` : la plus petite de `repeat` durées de `function(*args)`, mesurées avec `time.perf_counter()` (le minimum écarte les mesures ralenties par autre chose ; `timeit` fait la même chose).

La vérification compare les trois versions à `scipy.spatial.distance.cdist`, mesure chacune sur un morceau du problème et estime son temps pour les 500 × 2 000 distances, compte les `nan` de l'identité, puis classe les images de test par leur plus proche voisin avec tes distances. Ne lance pas la version par broadcasting sur tout le problème : la vérification affiche la mémoire qu'elle demanderait. Cet exercice se corrige avec `05_solutions.md`.

Dans tes notes : classe les trois versions par vitesse ; pourquoi le broadcasting n'est-il pas le plus rapide ? D'où viennent les `nan` de l'identité, et pourquoi n'apparaissent-ils qu'entre une image et elle-même (ou une image presque identique) ?

In [ ]:
TRAIN_34, Y_TRAIN_34 = wb.datasets.load_mnist("train", n=2000, flatten=True, seed=1334)
TEST_34, Y_TEST_34 = wb.datasets.load_mnist("test", n=500, flatten=True, seed=1334)
TRAIN_34, TEST_34 = TRAIN_34 / 255.0, TEST_34 / 255.0     # pixels in [0, 1], float64
print(TRAIN_34.shape, TEST_34.shape)

In [ ]:
def distances_loop_34(A, B):
    """Euclidean distance matrix (len(A), len(B)), with two Python loops over the rows (NumPy for one pair)."""
    raise NotImplementedError("distances_loop_34")


def distances_broadcast_34(A, B):
    """The same, with one broadcast difference A[:, None, :] - B[None, :, :] and no Python loop."""
    raise NotImplementedError("distances_broadcast_34")


def distances_dot_34(A, B):
    """The same, with ||a - b||² = ||a||² - 2 a.b + ||b||² (one matrix product for all the a.b), never nan."""
    raise NotImplementedError("distances_dot_34")


def timed_34(function, *args, repeat=3):
    """The smallest of `repeat` durations of function(*args), in seconds (time.perf_counter)."""
    raise NotImplementedError("timed_34")

In [ ]:
def as_matrix_34(name, D, shape):
    """The distance matrix returned by `name`, or None with a ❌ line when it is None or has the wrong shape."""
    if not returned("13.34", name, D):
        return None
    try:
        D = np.asarray(D, dtype=float)
    except (TypeError, ValueError):
        print(f"❌ Ex 13.34 : {name} doit rendre un tableau de nombres, pas un {type(D).__name__}.")
        return None
    if D.shape != shape:
        print(f"❌ Ex 13.34 : {name} doit rendre une matrice de forme {shape} (une ligne par ligne de A, une colonne "
              f"par ligne de B), pas {D.shape}.")
        return None
    return D


with checked("13.34"):
    A_34, B_34 = TEST_34[:10], TRAIN_34[:200]
    exact_34 = cdist(A_34, B_34)
    for name_34, function_34 in [("distances_loop_34", distances_loop_34), ("distances_broadcast_34", distances_broadcast_34),
                                 ("distances_dot_34", distances_dot_34)]:
        try:
            D_34 = as_matrix_34(name_34, function_34(A_34, B_34), (10, 200))
        except NotImplementedError:
            print(f"⏳ Ex 13.34 : {name_34} n'est pas encore écrite.")
            continue
        if D_34 is not None:
            verdict("13.34", np.allclose(D_34, exact_34), f"{name_34} rend les distances de scipy (cdist).",
                    f"{name_34}(TEST_34[:10], TRAIN_34[:200]) a la bonne forme, mais pas les distances de cdist "
                    f"(par exemple {D_34[0, 0]:.4f} au lieu de {exact_34[0, 0]:.4f} pour la première case) : la racine "
                    "de la somme des carrés des différences ?")
with checked("13.34"):
    calls_34 = []
    def slow_first_34():                                  # 0.1 s at the first call, then 0.01 s
        time.sleep(0.1 if not calls_34 else 0.01)
        calls_34.append(1)
    probe_34 = timed_34(slow_first_34, repeat=3)
    if returned("13.34", "timed_34", probe_34):
        if not isinstance(probe_34, (int, float, np.floating)):
            print(f"❌ Ex 13.34 : timed_34 doit rendre UNE durée (un nombre de secondes), pas un {type(probe_34).__name__}.")
        else:
            verdict("13.34", 0.009 < probe_34 < 0.02,
                    "timed_34 rend bien la plus petite des durées, en secondes.",
                    f"timed_34 rend {probe_34!r} pour trois appels de 0,1 s, 0,01 s et 0,01 s : la plus petite durée, "
                    "en secondes (environ 0,01), était attendue (pas la moyenne, ni la somme, ni des millisecondes).")
with checked("13.34"):
    pairs_full_34 = len(TEST_34) * len(TRAIN_34)               # 500 x 2,000 distances
    rows_34 = []
    for name_34, function_34, a_34, b_34 in [("loop", distances_loop_34, TEST_34[:10], TRAIN_34[:200]),
                                              ("broadcast", distances_broadcast_34, TEST_34[:50], TRAIN_34[:500]),
                                              ("dot product", distances_dot_34, TEST_34, TRAIN_34)]:
        seconds_34 = float(timed_34(function_34, a_34, b_34))
        rows_34.append((name_34, f"{len(a_34)} x {len(b_34)}", seconds_34, seconds_34 * pairs_full_34 / (len(a_34) * len(b_34))))
    fastest_34 = min(row[3] for row in rows_34)
    print(f"{'version':12s} {'measured on':12s} {'time (s)':>9s} {'500 x 2000 (s)':>15s} {'x fastest':>10s}")
    for name_34, size_34, seconds_34, full_34 in rows_34:
        print(f"{name_34:12s} {size_34:12s} {seconds_34:9.4f} {full_34:15.3f} {full_34 / fastest_34:10.0f}")
    print(f"memory of the broadcast difference for 500 x 2000 images: {500 * 2000 * 784 * 8 / 1e9:.1f} GB")
with checked("13.34"):
    with np.errstate(invalid="ignore"):
        squared_34 = ((TRAIN_34 ** 2).sum(axis=1)[:, None] - 2 * TRAIN_34 @ TRAIN_34.T
                      + (TRAIN_34 ** 2).sum(axis=1)[None, :])
        naive_nan_34 = int(np.isnan(np.sqrt(squared_34)).sum())
        mine_34 = as_matrix_34("distances_dot_34", distances_dot_34(TRAIN_34, TRAIN_34), (2000, 2000))
    if mine_34 is not None:
        mine_nan_34 = int(np.isnan(mine_34).sum())
        print(f"distances between the 2,000 training images: {naive_nan_34} nan with a plain sqrt of the identity, "
              f"{mine_nan_34} with distances_dot_34")
        verdict("13.34", mine_nan_34 == 0, "distances_dot_34 ne rend jamais nan.",
                "distances_dot_34 rend des nan : l'arrondi donne des carrés légèrement négatifs, dont la racine "
                "n'existe pas.")
        nearest_34 = np.argmin(as_matrix_34("distances_dot_34", distances_dot_34(TEST_34, TRAIN_34), (500, 2000)), axis=1)
        accuracy_34 = float(np.mean(Y_TRAIN_34[nearest_34] == Y_TEST_34))
        oracle_34 = KNeighborsClassifier(n_neighbors=1, algorithm="brute").fit(TRAIN_34, Y_TRAIN_34).score(TEST_34, Y_TEST_34)
        verdict("13.34", np.isclose(accuracy_34, oracle_34),
                f"le plus proche voisin calculé avec tes distances fait {fr(accuracy_34)}, comme scikit-learn.",
                f"le plus proche voisin calculé avec tes distances fait {fr(accuracy_34)}, scikit-learn {fr(oracle_34)}.")

### Ex 13.35 — Comparatif mylearn contre scikit-learn : accuracy et temps 🔬 ★★★ ⏱️ 45 min
**Objectif :** comparer des implémentations sur les mêmes données, en accuracy et en temps.  
**Prérequis :** Ex 13.14, 13.24 et 13.28 · ch. 12 (PCA) · fiche §13.7 · **Fil rouge :** MNIST · **Parcours :** C

500 images de MNIST pour l'entraînement et 500 pour le test, réduites à 50 coordonnées par une PCA (ch. 12). Écris `timed_fit_score_35(model, X_train, y_train, X_test, y_test)`, qui entraîne le modèle, mesure son accuracy sur le test, et rend le tuple `(accuracy, durée du fit en s, durée du score en s)`, mesurées avec `time.perf_counter()`.

La vérification l'applique à trois classifieurs de **ta** librairie et à leurs équivalents de scikit-learn : le kNN ($k = 3$), Naïve Bayes gaussien et l'arbre sans limite, affiche un tableau, contrôle les accuracies de ton kNN et de ton Naïve Bayes (a et b) et vérifie que ton arbre est à 3 points de celui de scikit-learn (les égalités de gain, fréquentes ici, se départagent autrement).

Dans tes notes : qui passe son temps à l'entraînement, qui à la prédiction ? Combien de fois ta librairie est-elle plus lente que scikit-learn pour chacun, et pourquoi (pense à ce que scikit-learn écrit en C, et à la boucle sur les seuils de ton `best_split`) ? Lequel des trois choisirais-tu sur ces données ?

In [ ]:
X_RAW_TRAIN_35, Y_TRAIN_35 = wb.datasets.load_mnist("train", n=500, flatten=True, seed=1335)
X_RAW_TEST_35, Y_TEST_35 = wb.datasets.load_mnist("test", n=500, flatten=True, seed=1335)
PCA_35 = PCA(n_components=50, random_state=0).fit(X_RAW_TRAIN_35 / 255.0)
Z_TRAIN_35, Z_TEST_35 = PCA_35.transform(X_RAW_TRAIN_35 / 255.0), PCA_35.transform(X_RAW_TEST_35 / 255.0)
print(Z_TRAIN_35.shape, Z_TEST_35.shape, "(500 images, 50 PCA coordinates)")

In [ ]:
def timed_fit_score_35(model, X_train, y_train, X_test, y_test):
    """Fit `model` on the training part, then score it on the test part. Return the tuple (accuracy, fit time in
    seconds, scoring time in seconds), both times measured with time.perf_counter."""
    raise NotImplementedError("timed_fit_score_35")

In [ ]:
def measured_35(result):
    """The tuple (accuracy, fit seconds, score seconds) returned by timed_fit_score_35, or None with a ❌ line."""
    if not returned("13.35", "timed_fit_score_35", result):
        return None
    if not isinstance(result, tuple) or len(result) != 3:
        print("❌ Ex 13.35 : timed_fit_score_35 doit rendre un tuple de trois nombres (accuracy, durée du fit, durée "
              f"du score), pas {type(result).__name__} {result!r}.")
        return None
    accuracy, fit_seconds, score_seconds = (float(value) for value in result)
    if not 0 <= accuracy <= 1:
        print(f"❌ Ex 13.35 : une accuracy de {accuracy} : elle est entre 0 et 1, en première position du tuple.")
        return None
    return accuracy, fit_seconds, score_seconds


pairs_35 = [("kNN (k = 3)", "13.14", lambda: mylearn.neighbors.KNeighborsClassifier(n_neighbors=3),
             lambda: KNeighborsClassifier(n_neighbors=3, algorithm="brute")),
            ("GaussianNB", "13.28", lambda: mylearn.naive_bayes.GaussianNB(), lambda: GaussianNB()),
            ("tree", "13.24", lambda: mylearn.tree.DecisionTreeClassifier(), lambda: DecisionTreeClassifier(random_state=0))]
results_35 = {}
with checked("13.35"):
    print(f"{'classifier':14s} {'library':13s} {'accuracy':>9s} {'fit (s)':>9s} {'score (s)':>10s}")
    for name_35, exercise_35, mine_35, oracle_35 in pairs_35:
        if not written(lambda: mine_35().fit(Z_TRAIN_35[:5], Y_TRAIN_35[:5])):
            print(f"⏳ {name_35} : ta classe ({exercise_35}) n'est pas encore écrite.")
            continue
        for library_35, model_35 in [("mylearn", mine_35()), ("scikit-learn", oracle_35())]:
            try:
                result_35 = measured_35(timed_fit_score_35(model_35, Z_TRAIN_35, Y_TRAIN_35, Z_TEST_35, Y_TEST_35))
            except AttributeError as exc:
                if "NoneType" not in str(exc):
                    raise
                print(f"❌ Ex 13.35 : {name_35} ({library_35}) : {exc} : n'enchaîne pas fit(...).score(...) (un fit "
                      "sans return self rend None), appelle fit, puis score.")
                result_35 = None
            if result_35 is None:
                break
            results_35[name_35, library_35] = result_35
            print(f"{name_35:14s} {library_35:13s} {result_35[0]:9.3f} {result_35[1]:9.3f} {result_35[2]:10.3f}")
if ("kNN (k = 3)", "mylearn") in results_35:
    wb.check("13.35a", results_35["kNN (k = 3)", "mylearn"][0], computed=True)
if ("GaussianNB", "mylearn") in results_35:
    wb.check("13.35b", results_35["GaussianNB", "mylearn"][0], computed=True)
if ("tree", "mylearn") in results_35 and ("tree", "scikit-learn") in results_35:
    tree_gap_35 = abs(results_35["tree", "mylearn"][0] - results_35["tree", "scikit-learn"][0])
    verdict("13.35", tree_gap_35 <= 0.03,
            "ton arbre a l'accuracy de celui de scikit-learn, à 3 points près (les égalités de gain se départagent "
            "autrement).",
            f"ton arbre s'écarte de {fr(tree_gap_35)} de celui de scikit-learn : plus de 3 points, ce n'est pas qu'une "
            "affaire d'égalités ; relance les tests de 13.24.")

### Ex 13.36 — Défi : 96 % sur MNIST avec un classifieur classique 🏆 ★★★★ ⏱️ 100 min
**Objectif :** construire, régler et valider un classifieur classique qui atteint 96 % sur MNIST.  
**Prérequis :** Ex 13.18, 13.14 · ch. 12 (PCA, mise à l'échelle) · ch. 8 (validation croisée) · **Fil rouge :** MNIST · **Parcours :** C

**Objectif** : au moins **96 %** d'accuracy sur les 2 000 premières images de test de MNIST, avec au plus **20 000** images d'entraînement, sans réseau de neurones : un classifieur de ce chapitre (kNN, SVM, arbre, Naïve Bayes, régression logistique), avec la préparation du ch. 12 si tu veux (PCA, mise à l'échelle). Écris `N_TRAIN_36`, le nombre d'images d'entraînement, et `build_model_36()`, qui rend un modèle scikit-learn **neuf** (souvent un pipeline) qui reçoit des pixels dans $[0, 1]$.

Règles : les 2 000 images de test ne servent qu'à la mesure finale ; règle tes hyperparamètres par validation croisée sur l'entraînement (dans une cellule à toi, sur un sous-ensemble si c'est long). La vérification charge `N_TRAIN_36` images d'entraînement, entraîne ton modèle, mesure les temps et l'accuracy. Ce défi se corrige avec `05_solutions.md`.

Dans tes notes : les modèles essayés, leurs réglages et leurs scores de validation croisée ; le temps d'entraînement et de prédiction du modèle retenu ; ce que tu essaierais pour aller plus loin.

In [ ]:
X_TEST_36, Y_TEST_36 = wb.datasets.load_mnist("test", flatten=True)
X_TEST_36, Y_TEST_36 = X_TEST_36[:2000] / 255.0, Y_TEST_36[:2000]       # the first 2,000 test images, never for tuning
print(X_TEST_36.shape)

In [ ]:
N_TRAIN_36 = ...   # the number of training images you use (at most 20,000)


def build_model_36():
    """Return a NEW, unfitted scikit-learn classifier (often a pipeline) that takes pixels in [0, 1], shape (n, 784).
    No neural network: a classifier of this chapter, with any preprocessing of chapter 12."""
    raise NotImplementedError("build_model_36")

In [ ]:
with wb.attempt("13.36"):
    if not filled(N_TRAIN_36):
        raise NotImplementedError("N_TRAIN_36")
    if not 1 <= int(N_TRAIN_36) <= 20_000:
        print(f"❌ Ex 13.36 : {N_TRAIN_36} images d'entraînement : le défi en demande de 1 à 20 000.")
    else:
        X_train_36, y_train_36 = wb.datasets.load_mnist("train", n=int(N_TRAIN_36), flatten=True, seed=1336)
        model_36 = build_model_36()
        if returned("13.36", "build_model_36", model_36):
            start_36 = time.perf_counter()
            model_36.fit(X_train_36 / 255.0, y_train_36)
            fit_36 = time.perf_counter() - start_36
            start_36 = time.perf_counter()
            accuracy_36 = model_36.score(X_TEST_36, Y_TEST_36)
            print(f"{int(N_TRAIN_36)} training images · fit {fit_36:.1f} s · prediction of 2,000 images "
                  f"{time.perf_counter() - start_36:.1f} s · test accuracy {accuracy_36:.4f}")
            verdict("13.36", accuracy_36 >= 0.96, f"{fr(100 * accuracy_36, 2)} % : défi réussi.",
                    f"{fr(100 * accuracy_36, 2)} % : il manque {fr(100 * (0.96 - accuracy_36), 2)} point(s) ; règle ton "
                    "modèle par validation croisée sur l'entraînement, jamais sur ces 2 000 images.")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dire, pour chaque classifieur du chapitre, ce qu'il garde en mémoire, où il passe son temps et ce qu'il suppose des données ?
2. Sais-tu calculer à la main un vote kNN, une marge, un gain de split et un posterior de Naïve Bayes ?
3. Sais-tu choisir une baseline et régler ses hyperparamètres par validation croisée ?

**Pour aller plus loin** : les guides utilisateur de scikit-learn sur les plus proches voisins, les SVM, les arbres et Naïve Bayes, et l'article de Domingos et Pazzani, cités dans la fiche. La suite : le ch. 14 (ensembles), qui construira forêts aléatoires, AdaBoost et gradient boosting avec tes arbres de `mylearn.tree`.